In [ ]:
# Libraries used in this notebook
import pandas as pd  # tables (DataFrames) for all data handling
import numpy as np  # numerical arrays and maths
import requests  # web requests (OpenStreetMap Nominatim, OSRM routing)
import time  # pauses between web requests (rate limits)
from pathlib import Path  # file paths that work on every operating system
from pyproj import Geod  # distances between GPS points on the earth (WGS84)

from scipy.optimize import milp, Bounds, LinearConstraint  # integer optimisation solver for the bus-model assignment
from scipy.sparse import lil_matrix  # sparse matrix to build the optimisation constraints

In [ ]:
#inladen van alle benodigde databestanden uit GTFS file en filteren van de data zodat we gegevens per dag krijgen
# GTFS folder
gtfs_dir = Path("gtfs-nl")  # folder with the unzipped GTFS feed (gtfs.ovapi.nl)

# Representative working day
representative_date = pd.Timestamp("2026-11-04")  # a normal weekday (Wednesday) inside the feed period
datum_gtfs = representative_date.strftime("%Y%m%d")  # same date as text YYYYMMDD, the format GTFS uses

# Loading GTFS datasets
routes = pd.read_csv(gtfs_dir / "routes.txt", dtype=str)  # lines: route_id, line number, name, operator
trips_all = pd.read_csv(gtfs_dir / "trips.txt", dtype=str)  # all trips of all days: trip_id, route_id, service_id, shape_id
stops = pd.read_csv(gtfs_dir / "stops.txt", dtype=str)  # stops with names and coordinates
stop_times = pd.read_csv(gtfs_dir / "stop_times.txt", dtype=str)  # time of every trip at every stop (large file)
shapes = pd.read_csv(gtfs_dir / "shapes.txt", dtype=str)  # GPS points that draw the route of each trip

# Service calendar: on which dates each service_id runs (read as text)
calendar_dates = pd.read_csv(
    gtfs_dir / "calendar_dates.txt",
    dtype=str
)

# Cleaning relevant columns
calendar_dates["date"] = calendar_dates["date"].str.strip()  # remove spaces around the dates
calendar_dates["exception_type"] = (  # remove spaces (1 = service added, 2 = removed)
    calendar_dates["exception_type"].str.strip()
)
calendar_dates["service_id"] = (  # remove spaces around the service IDs
    calendar_dates["service_id"].str.strip()
)

# Find services for the selected day
exceptions = calendar_dates[  # keep only the calendar rows of the selected date
    calendar_dates["date"] == datum_gtfs
].copy()

# Show what was found for this date (check)
print("Selected date:", representative_date.date())
print("Calendar records on date:", len(exceptions))  # print a check / summary line
print("Exception types:")  # print a check / summary line
print(exceptions["exception_type"].value_counts())  # print a check / summary line

# Active services in a calendar_dates-only feed
active_services = set(  # service_ids that run on this date (exception_type 1)
    exceptions.loc[
        exceptions["exception_type"] == "1",
        "service_id"
    ]
)

# Exclude removed services
active_services -= set(  # minus service_ids cancelled on this date (exception_type 2)
    exceptions.loc[
        exceptions["exception_type"] == "2",
        "service_id"
    ]
)

print("Active services:", len(active_services))  # number of services running that day

# Stop with a clear error if no service runs on the chosen date
if not active_services:
    raise ValueError(  # stop with an error message
        "No active services found for the selected date. "
        "Check calendar_dates.txt."
    )

# Select trips belonging to active services
trips = trips_all[  # trips whose service runs on the selected date
    trips_all["service_id"].str.strip().isin(active_services)
].drop_duplicates("trip_id").copy()  # each trip once; copy so trips_all is not changed

print("Total trips in GTFS:", len(trips_all))  # all trips in the feed
print("Trips on selected day:", len(trips))  # trips on the selected day

# Stop if no trips are left after filtering
if trips.empty:
    raise ValueError(  # stop with an error message
        "No trips match the active services on this date."
    )

# Loading fleet information from the central Excel file
df_vloot_input = pd.read_excel(  # fleet per bus model (sheet Blad1, headers in the 2nd row)
    "Distribution of the busses in Utrecht.xlsx",
    sheet_name="Blad1",
    header=1
)

# Remove whitespace from column names
df_vloot_input.columns = (  # column names without leading/trailing spaces
    df_vloot_input.columns.astype(str).str.strip()
)

# Display fleet data
display(df_vloot_input[  # show maker, model, number, kWh/km and battery of the first 10 models
    ["Company", "type", "adjusted number",
     "Average independent source", "Max. capacity (kWh)"]
].head(10))

# Summary of this step
print("GTFS data successfully filtered.")

print("Selected date:", representative_date.date())  # print a check / summary line
print("Active services:", len(active_services))  # print a check / summary line
print("Selected trips:", trips["trip_id"].nunique())  # number of unique trips on that day


Selected date: 2026-11-04
Calendar records on date: 984
Exception types:
exception_type
1    984
Name: count, dtype: int64
Active services: 984
Total trips in GTFS: 703101
Trips on selected day: 124179


,Company,type,adjusted number,Average independent source,Max. capacity (kWh)
0,VDL,Citea LF-122 Electric,17.0,0.818182,450.00
1,Yutong,U12 LE,27.0,0.890000,465.99
2,Yutong,U18 LF,48.0,1.000000,704.17
3,Solaris,Urbino 9 Electric,13.0,0.800000,350.00
4,Solaris,Urbino 15 Electric,16.0,1.880000,470.00
5,Solaris,Urbino 18 Electric,52.0,2.215000,443.00
6,Solaris,Urbino 24 Electric,25.0,2.707692,704.00
7,Volvo,7900 EA,52.0,1.000000,316.00
8,adopted from previous conseccion,NaN,NaN,NaN,NaN
9,Iveco,E-way 10.7,3.0,0.920000,385.00


GTFS data successfully filtered.
Selected date: 2026-11-04
Active services: 984
Selected trips: 124179


In [ ]:
# Calculate the length of each route shape

geod = Geod(ellps="WGS84")  # earth model for distances between GPS points

shapes["shape_pt_lat"] = pd.to_numeric(shapes["shape_pt_lat"])  # latitude text -> number
shapes["shape_pt_lon"] = pd.to_numeric(shapes["shape_pt_lon"])  # longitude text -> number
shapes["shape_pt_sequence"] = pd.to_numeric(shapes["shape_pt_sequence"])  # point order text -> number

lengths = {}  # dictionary: shape_id -> route length in km

for shape_id, points in shapes.groupby("shape_id"):  # loop over every route shape
    points = points.sort_values("shape_pt_sequence")  # put the points in driving order
    lat = points["shape_pt_lat"].to_list()  # latitudes as a list
    lon = points["shape_pt_lon"].to_list()  # longitudes as a list

    _, _, distances = geod.inv(lon[:-1], lat[:-1], lon[1:], lat[1:])  # distance (m) from each point to the next one
    # Route length = sum of all small distances
    lengths[shape_id] = sum(distances) / 1000   # convert meters to kilometers

In [ ]:
# Calculate the planned duration of each trip

stop_times["stop_sequence"] = pd.to_numeric(stop_times["stop_sequence"], errors="coerce")  # stop order text -> number (invalid -> NaN)

stop_times = stop_times.dropna(subset=["stop_sequence"])  # drop rows without a valid stop order
stop_times = stop_times.sort_values(["trip_id", "stop_sequence"])  # stops of each trip in driving order

begin = stop_times.groupby("trip_id").first().reset_index()  # first stop of each trip (departure)
end = stop_times.groupby("trip_id").last().reset_index()  # last stop of each trip (arrival)

In [ ]:
# Convert GTFS-times to seconds

# Function: 'HH:MM:SS' -> seconds after midnight (GTFS times can be later than 24:00)
def time_to_seconds(tijd):
    if pd.isna(tijd):  # missing time
        return None  # -> no value
    hours, minutes, seconds = map(int, tijd.split(":"))  # split the text and turn the parts into integers
    return hours * 3600 + minutes * 60 + seconds  # total number of seconds

begin["begin_seconds"] = begin["departure_time"].map(time_to_seconds)  # departure of each trip in seconds
end["eind_seconds"] = end["arrival_time"].map(time_to_seconds)  # arrival of each trip in seconds

duration_per_trip = begin[["trip_id", "begin_seconds"]].merge(end[["trip_id", "eind_seconds"]], on="trip_id")  # one row per trip with start and end time

duration_per_trip["duration_minutes"] = (duration_per_trip["eind_seconds"] - duration_per_trip["begin_seconds"]) / 60  # duration = (end - start) / 60 minutes

In [ ]:
# Linking line info, distance and duration to each trip

# Add the line information (routes) and the duration to every trip of the day
ritten = trips.merge(routes, on="route_id", how="left").merge(duration_per_trip[["trip_id", "duration_minutes"]], on="trip_id", how="left")

ritten["length_km"] = ritten["shape_id"].map(lengths)  # route length (km) of each trip via its shape

In [ ]:
# Remove trips with unusable values

ritten = ritten[pd.to_numeric(ritten["route_type"], errors="coerce") == 3].copy()  # keep only buses (GTFS route_type 3)

ritten = ritten.dropna(subset=["length_km", "duration_minutes"])  # drop trips without length or duration
ritten = ritten[ritten["duration_minutes"] >= 0]  # drop trips with a negative duration (data errors)

In [ ]:
#Sumarize per line, direction and destination

# Grouping columns (only the ones that exist in the data)
group = [col for col in ["route_short_name", "route_long_name", "direction_id", "trip_headsign", "agency_id"] if col in ritten.columns]
# Per group: shortest/average/longest km and minutes, and the number of trips
result = ritten.groupby(group).agg(shortest_km=("length_km", "min"), average_km=("length_km", "mean"), longest_km=("length_km", "max"), shortest_minutes=("duration_minutes", "min"), average_minutes=("duration_minutes", "mean"), longest_minutes=("duration_minutes", "max"), amount_trips=("trip_id", "nunique")).reset_index()

In [ ]:
# Selecting U-OV and Transdev bus services, UOV for not updated routes and TRANSDEV for updated
uov_result = result[  # keep the U-OV / Transdev lines
    result["agency_id"].isin(["UOV", "TRANSDEV"])
].copy()

In [ ]:
# Rounding to 2 decimals

km_columns = ["shortest_km", "average_km", "longest_km"]  # distance columns
minutes_columns = ["shortest_minutes", "average_minutes", "longest_minutes"]  # duration columns

uov_result[km_columns] = uov_result[km_columns].round(2)  # round distances to 2 decimals
uov_result[minutes_columns] = uov_result[minutes_columns].round(2)  # round durations to 2 decimals

In [ ]:
#Display preliminary GTFS line statistics

display(uov_result)  # table per line, direction and destination

,route_short_name,route_long_name,direction_id,trip_headsign,agency_id,shortest_km,average_km,longest_km,shortest_minutes,average_minutes,longest_minutes,amount_trips
25,1,Hilversum Station - Hilversum Kerkelanden,0,Kerkelanden,TRANSDEV,10.59,10.59,10.59,26.0,31.76,38.0,62
52,1,Overvecht - Utrecht CS - Hoograven,0,Hoograven,UOV,12.45,12.82,13.40,37.0,47.92,53.0,71
53,1,Overvecht - Utrecht CS - Hoograven,1,Overvecht,UOV,12.62,12.86,13.33,37.0,47.13,51.0,71
80,1,U-bus Soesterkwartier - Amersfoort CS,0,Centraal Station,UOV,2.66,2.66,2.66,9.0,9.00,9.0,1
81,1,U-bus Soesterkwartier - Amersfoort CS,0,Isselt,UOV,4.12,4.12,4.12,12.0,14.00,15.0,18
82,1,U-bus Soesterkwartier - Amersfoort CS,0,Soesterkwartier,UOV,5.70,5.70,5.70,18.0,18.38,19.0,13
83,1,U-bus Soesterkwartier - Amersfoort CS,1,Centraal Station,UOV,3.89,4.82,5.54,11.0,13.75,18.0,32
105,10,Lunetten - Leidsche Rijn,0,Leidsche Rijn,UOV,9.28,9.28,9.28,23.0,25.69,29.0,26
106,10,Lunetten - Leidsche Rijn,1,Lunetten,UOV,9.24,9.24,9.24,23.0,26.85,29.0,26
129,100,Bussum - Huizen - Hilversum Station Sportpark,0,Blaricum P+R,TRANSDEV,15.38,15.38,15.38,35.0,36.20,39.0,10


In [ ]:

# STEP 13: Combining bus directions using daily GTFS data

import pandas as pd  # already imported above; harmless

# 1. Load bus line requirements from the central Excel file
bus_data = pd.read_excel(  # buses needed, peak frequency and possible models per line and direction
    "Distribution of the busses in Utrecht.xlsx",
    sheet_name="Bus line requirements"
)

# 2. Standardise bus line numbers
# Function: write every line number the same way ('1.0' -> '1', 'n41' -> 'N41')
def normalise_line_number(line):
    if pd.isna(line):  # empty cell
        return None  # -> no line number

    line = str(line).strip()  # as text without spaces

    if not line:  # empty text
        return None  # -> no line number

    try:  # try to read it as a number
        number = float(line)  # e.g. '12' or '12.0' -> 12.0
        if number.is_integer():  # whole number
            return str(int(number))  # -> '12'
    except ValueError:  # not a number (e.g. 'N41')
        pass  # keep it as text

    return line.upper()  # text line numbers in capitals


# 3. Standardise route descriptions
# Function: write route names the same way (lower case, one kind of dash), so they can be matched
def normalise_route_name(name):
    if pd.isna(name):  # empty cell
        return None  # -> no name

    return (  # give back the result
        str(name)  # as text
        .casefold()  # lower case
        .replace("–", "-")  # en dash -> normal dash
        .replace("—", "-")  # em dash -> normal dash
        .replace(" - ", "-")  # no spaces around dashes
        .strip()  # no spaces at the start and end
    )


# 4. Prepare the operational dataset
bus_data["route_short_name"] = (  # uniform line numbers
    bus_data["route_short_name"].apply(normalise_line_number)
)

bus_data["route_name_clean"] = (  # uniform route names (for matching with GTFS)
    bus_data["route_long_name"].apply(normalise_route_name)
)

# Remove rows without line numbers
missing_line_numbers = bus_data[  # rows without a line number
    bus_data["route_short_name"].isna()
].copy()

if not missing_line_numbers.empty:  # if there are any: show them
    print("WARNING: Rows without a valid line number:")  # print a check / summary line
    display(missing_line_numbers)  # show the table

bus_data = bus_data.dropna(  # and remove them
    subset=["route_short_name"]
).copy()

# Exclude night bus lines
bus_data_day = bus_data[  # day lines only: drop night lines (numbers starting with N)
    ~bus_data["route_short_name"]
    .str.upper().str.startswith("N")
].copy()


# 5. Prepare GTFS route information
gtfs_routes = routes.copy()  # copy of the GTFS lines

gtfs_routes["route_short_name"] = (  # uniform line numbers
    gtfs_routes["route_short_name"]
    .apply(normalise_line_number)
)

gtfs_routes["route_name_clean"] = (  # uniform route names
    gtfs_routes["route_long_name"]
    .apply(normalise_route_name)
)

# Only relevant operators
gtfs_routes = gtfs_routes[  # keep operators UOV and TRANSDEV
    gtfs_routes["agency_id"].isin(["UOV", "TRANSDEV"])
].copy()

# Only bus routes
gtfs_routes = gtfs_routes[  # keep only bus routes (route_type 3)
    pd.to_numeric(
        gtfs_routes["route_type"],
        errors="coerce"
    ) == 3
].copy()


# 6. Match GTFS routes with Utrecht Binnen requirements
approved_routes = gtfs_routes.merge(  # GTFS routes whose line number AND name are in the Utrecht Binnen table
    bus_data_day[
        ["route_short_name", "route_name_clean"]
    ].dropna().drop_duplicates(),
    on=["route_short_name", "route_name_clean"],
    how="inner",
    validate="many_to_many"
)

approved_routes = approved_routes.drop_duplicates(  # each GTFS route_id once
    subset=["route_id"]
).copy()

approved_route_ids = set(  # set of the approved route IDs
    approved_routes["route_id"]
)

print("Approved GTFS routes:", len(approved_route_ids))  # check


# 7. Select daily GTFS trips from approved routes
gtfs_bus = ritten[  # trips of the day on the approved routes
    ritten["route_id"].isin(approved_route_ids)
].copy()

# Standardise line numbers
gtfs_bus["route_short_name"] = (  # uniform line numbers
    gtfs_bus["route_short_name"]
    .apply(normalise_line_number)
)

# Exclude night services
gtfs_bus = gtfs_bus[  # drop night services
    ~gtfs_bus["route_short_name"]
    .fillna("")
    .str.upper().str.startswith("N")
].copy()

print("Selected daily trips:", gtfs_bus["trip_id"].nunique())  # number of trips that day
print(  # print a check / summary line
    "Selected bus lines:",
    gtfs_bus["route_short_name"].nunique()
)


# 8. Calculate daily GTFS statistics per bus line
gtfs_line = (  # statistics per line (both directions together)
    gtfs_bus.groupby("route_short_name", as_index=False)
    .agg(
        shortest_km=("length_km", "min"),  # shortest trip (km)
        average_km=("length_km", "mean"),  # average trip (km)
        longest_km=("length_km", "max"),  # longest trip (km)
        shortest_time=("duration_minutes", "min"),  # shortest trip (min)
        average_time=("duration_minutes", "mean"),  # average trip (min)
        longest_time=("duration_minutes", "max"),  # longest trip (min)
        aantal_ritten=("trip_id", "nunique")  # number of trips that day
    )
)


# 9. Combine eligible bus models
# Function: all bus models of a line (both directions) as one text 'A; B; C'
def combine_models(group):
    models = set()  # empty set (no duplicates)

    for value in group["Modellen"].dropna():  # every non-empty model cell
        for model in str(value).split(";"):  # split 'A; B' into separate models
            model = model.strip()  # without spaces

            if model:  # skip empty parts
                models.add(model)  # add to the set

    return "; ".join(sorted(models))  # sorted, as one text


# 10. Combine operational data for both directions
operational_data = []  # list that gets one dict per line

for line, group in bus_data_day.groupby("route_short_name"):  # loop over every day line

    operational_data.append({  # add one row to the list
        "route_short_name": line,
        "Aantal bussen": group["Aantal bussen"].sum(),  # buses of both directions added up
        "Spitsfrequentie (p.u.)":  # highest peak frequency of the two directions
            group["Spitsfrequentie (p.u.)"].max(),
        "Modellen": combine_models(group)  # all possible models
    })

operational_data = pd.DataFrame(operational_data)  # list -> table


# 11. Merge operational data with daily GTFS statistics
bus_lines = operational_data.merge(  # one row per line: buses + GTFS statistics
    gtfs_line,
    on="route_short_name",
    how="left",
    validate="one_to_one"
)


# 12. Check for missing GTFS data
missing = bus_lines[  # lines without GTFS trips on that day
    bus_lines["aantal_ritten"].isna()
].copy()

if not missing.empty:  # if any line is missing:
    print("WARNING: Regular lines without daily GTFS data:")  # print a check / summary line
    display(missing)  # show the table

    raise ValueError(  # stop: the matching must be fixed first
        "Some regular bus lines have no matching daily GTFS data. "
        "Check route names and line numbers."
    )


# 13. Calculate average speed
bus_lines["average_speed"] = (  # km/h = average km / (average minutes / 60)
    bus_lines["average_km"]
    / (bus_lines["average_time"] / 60)
)


# 14. Sort bus lines numerically
bus_lines["sort_number"] = pd.to_numeric(  # line number as a number, for sorting (text -> NaN)
    bus_lines["route_short_name"],
    errors="coerce"
)

bus_lines = (  # sort by line number and drop the helper column
    bus_lines.sort_values("sort_number")
    .drop(columns="sort_number")
    .reset_index(drop=True)
)


# 15. Display results
print("\nNumber of regular bus lines:", len(bus_lines))  # summary
print("Total required buses:", bus_lines["Aantal bussen"].sum())  # print a check / summary line
print("Total daily trips:", bus_lines["aantal_ritten"].sum())  # print a check / summary line

display(bus_lines)  # show the line table


# 16. Check line 1
print("\nResults for line 1:")

display(  # show the table
    bus_lines[
        bus_lines["route_short_name"] == "1"
    ]
)


# 17. Check approved routes
print("\nApproved routes for Utrecht Binnen:")

display(  # show the table
    approved_routes[
        ["route_id", "route_short_name", "route_long_name"]
    ].sort_values("route_short_name")
)


,route_short_name,direction_id,Spitsfrequentie (p.u.),Aantal bussen,Modellen,route_long_name,route_name_clean
110,NaN,NaN,444.875,316,NaN,NaN,NaN


Approved GTFS routes: 45
Selected daily trips: 3259
Selected bus lines: 44

Number of regular bus lines: 44
Total required buses: 311
Total daily trips: 3259


,route_short_name,Aantal bussen,Spitsfrequentie (p.u.),Modellen,shortest_km,average_km,longest_km,shortest_time,average_time,longest_time,aantal_ritten,average_speed
0,1,8,5.0000,7900 EA; Urbino 18 Electric,12.445404,12.841189,13.396250,37.0,47.521127,53.0,142,16.213239
1,2,1,2.0000,E-way 10.7,5.048149,5.048149,5.048149,19.0,22.346154,23.0,26,13.554411
2,3,13,5.0000,GX437 ELEC,5.083383,11.913150,12.188609,16.0,45.594595,52.0,148,15.677055
3,5,15,4.0000,7900 EA; Citea LF-122 Electric; U12 LE; Urbino...,5.516617,15.077154,19.506721,19.0,49.551020,66.0,98,18.256521
4,6,9,4.0000,7900 EA; Citea LF-122 Electric; U12 LE; Urbino...,6.767892,12.990394,15.372934,23.0,43.492308,56.0,130,17.920954
5,7,8,5.0000,GX437 ELEC,11.459491,11.464402,11.469313,38.0,45.347222,50.0,144,15.168826
6,8,8,5.0000,GX437 ELEC,11.117236,11.132362,11.147069,34.0,40.542254,44.0,142,16.475200
7,9,4,8.0000,7900 EA; Urbino 18 Electric,3.305388,3.356049,3.409633,11.0,11.906542,14.0,107,16.911959
8,10,2,2.0000,Citea LF-122 Electric; U12 LE; Urbino 9 Electric,9.240024,9.259223,9.278422,23.0,26.269231,29.0,52,21.148446
9,11,4,2.0000,Citea LF-122 Electric; U12 LE; Urbino 9 Electric,5.333964,10.740238,10.953792,13.0,32.230769,42.0,52,19.993761



Results for line 1:


,route_short_name,Aantal bussen,Spitsfrequentie (p.u.),Modellen,shortest_km,average_km,longest_km,shortest_time,average_time,longest_time,aantal_ritten,average_speed
0,1,8,5.0,7900 EA; Urbino 18 Electric,12.445404,12.841189,13.39625,37.0,47.521127,53.0,142,16.213239



Approved routes for Utrecht Binnen:


,route_id,route_short_name,route_long_name
0,146869,1,Overvecht - Utrecht CS - Hoograven
8,146877,10,Lunetten - Leidsche Rijn
9,146878,11,Vleuten - Overvecht
39,146911,111,Vleuten - Breukelen
10,146879,12,Utrecht CS - Maarssen
40,146912,122,Utrecht - Nieuw Loosdrecht
41,146913,128,Maarssen - Vleuten
11,146880,13,Utrecht CS - Lage Weide - Maarssen
42,146914,158,Bilthoven - Berg en Bosch
12,146881,16,De Meern - Utrecht CS


In [ ]:

# Selecting approved Utrecht Binnen routes operating on the selected day

# Get route IDs with trips on the selected day
active_route_ids = set(trips["route_id"])  # routes with trips on the selected day

# Select approved routes with active trips
selected_routes = approved_routes[  # approved routes that also run that day
    approved_routes["route_id"].isin(active_route_ids)
].copy()

# Remove duplicate route IDs
selected_routes = selected_routes.drop_duplicates(  # each route once
    subset=["route_id"]
)

# Check the number of selected routes and bus lines
print("Selected GTFS routes:", len(selected_routes))

print(  # print a check / summary line
    "Selected bus lines:",
    selected_routes["route_short_name"].nunique()
)

# Check whether all bus lines are represented
missing_lines = (  # day lines without any running GTFS route
    set(bus_lines["route_short_name"])
    - set(selected_routes["route_short_name"])
)

print("Missing bus lines:", missing_lines)  # should be an empty set

if missing_lines:  # stop if a line is missing
    raise ValueError(  # stop with an error message
        f"Bus lines without matching GTFS routes: {sorted(missing_lines)}"
    )

# Display selected routes
display(
    selected_routes[
        ["route_id", "route_short_name", "route_long_name"]
    ].sort_values("route_short_name")
)


Selected GTFS routes: 45
Selected bus lines: 44
Missing bus lines: set()


,route_id,route_short_name,route_long_name
0,146869,1,Overvecht - Utrecht CS - Hoograven
8,146877,10,Lunetten - Leidsche Rijn
9,146878,11,Vleuten - Overvecht
39,146911,111,Vleuten - Breukelen
10,146879,12,Utrecht CS - Maarssen
40,146912,122,Utrecht - Nieuw Loosdrecht
41,146913,128,Maarssen - Vleuten
11,146880,13,Utrecht CS - Lage Weide - Maarssen
42,146914,158,Bilthoven - Berg en Bosch
12,146881,16,De Meern - Utrecht CS


In [1]:

# Matching GTFS routes to selected bus lines

# Use the approved routes from the previous steps
matched_routes = selected_routes[  # route ID, line number and name of the matched routes
    ["route_id", "route_short_name", "route_long_name"]
].drop_duplicates(subset=["route_id"]).copy()

# Check whether all bus lines are matched
unmatched_lines = (  # lines in the bus table without a matched route
    set(bus_lines["route_short_name"])
    - set(matched_routes["route_short_name"])
)

print("Number of matched GTFS routes:", len(matched_routes))  # print a check / summary line

print(  # print a check / summary line
    "Number of matched bus lines:",
    matched_routes["route_short_name"].nunique()
)

print("Unmatched bus lines:", unmatched_lines)  # print a check / summary line

# Stop if any bus lines are missing
if unmatched_lines:  # stop if any line is unmatched
    raise ValueError(  # stop with an error message
        f"Unmatched bus lines: {sorted(unmatched_lines)}"
    )

# Display matched routes
display(
    matched_routes.sort_values(
        ["route_short_name", "route_id"]
    )
)


In [ ]:

# Connect the matched Utrecht Binnen routes to daily GTFS trips

# Select trips belonging to the approved routes
selected_trips = trips.merge(  # trips of the day + their line number (matched routes only)
    matched_routes[
        ["route_id", "route_short_name"]
    ].drop_duplicates("route_id"),
    on="route_id",
    how="inner",
    validate="many_to_one"  # many trips per route, one line per route (checked by pandas)
)

# Ensure every trip occurs only once
selected_trips = selected_trips.drop_duplicates(  # each trip once
    subset=["trip_id"]
).copy()

# Check results
print("Selected trips:", selected_trips["trip_id"].nunique())

print(  # print a check / summary line
    "Unique bus lines:",
    selected_trips["route_short_name"].nunique()
)

# Check whether all regular bus lines have trips
missing_lines = (  # lines without trips
    set(bus_lines["route_short_name"])
    - set(selected_trips["route_short_name"])
)

print("Lines without trips:", missing_lines)  # print a check / summary line

if missing_lines:  # stop if a line has no trips
    raise ValueError(  # stop with an error message
        f"Bus lines without selected trips: {sorted(missing_lines)}"
    )

# Check bus lines with multiple GTFS route IDs
multiple_routes = matched_routes[  # lines with more than one GTFS route_id (route variants)
    matched_routes.duplicated(
        "route_short_name",
        keep=False
    )
].copy()

display(  # show the table
    multiple_routes[
        ["route_id", "route_short_name", "route_long_name"]
    ].sort_values("route_short_name")
)


Selected trips: 3259
Unique bus lines: 44
Lines without trips: set()


,route_id,route_short_name,route_long_name
2,146871,3,Utrecht CS - Overvecht / Zuilen - Utrecht CS
38,146910,3,Utrecht CS - Zuilen / Overvecht - Utrecht CS


In [ ]:

# Select stop times for the matched trips
selected_stops = stop_times[  # stop times of the selected trips only
    stop_times["trip_id"].isin(selected_trips["trip_id"])
].copy()

selected_stops["stop_sequence"] = pd.to_numeric(  # stop order as a number
    selected_stops["stop_sequence"]
)

# Identify first and last stops per trip
first_stops = selected_stops.loc[  # row with the lowest stop_sequence = first stop of each trip
    selected_stops.groupby("trip_id")["stop_sequence"].idxmin()
]

last_stops = selected_stops.loc[  # row with the highest stop_sequence = last stop of each trip
    selected_stops.groupby("trip_id")["stop_sequence"].idxmax()
]

# Check: one first and one last stop per trip
print("Trips:", len(selected_trips))
print("First stops:", len(first_stops))  # print a check / summary line
print("Last stops:", len(last_stops))  # print a check / summary line


Trips: 3259
First stops: 3259
Last stops: 3259


In [ ]:

# Select stop information
stop_info = stops[  # stop ID, name and coordinates
    ["stop_id", "stop_name", "stop_lat", "stop_lon"]
]

# Connect first and last stop IDs to each trip
trip_endpoints = selected_trips.merge(  # add the first stop ID to each trip
    first_stops[["trip_id", "stop_id"]].rename(
        columns={"stop_id": "first_stop_id"}
    ),
    on="trip_id",
    how="left"
)

trip_endpoints = trip_endpoints.merge(  # add the last stop ID
    last_stops[["trip_id", "stop_id"]].rename(
        columns={"stop_id": "last_stop_id"}
    ),
    on="trip_id",
    how="left"
)

# Add first stop coordinates
trip_endpoints = trip_endpoints.merge(  # add name and coordinates of the first stop (columns first_...)
    stop_info.add_prefix("first_"),
    left_on="first_stop_id",
    right_on="first_stop_id",
    how="left"
)

# Add last stop coordinates
trip_endpoints = trip_endpoints.merge(  # add name and coordinates of the last stop (columns last_...)
    stop_info.add_prefix("last_"),
    left_on="last_stop_id",
    right_on="last_stop_id",
    how="left"
)

# Convert coordinates to numeric values
for col in ["first_stop_lat", "first_stop_lon",  # the four coordinate columns
            "last_stop_lat", "last_stop_lon"]:
    trip_endpoints[col] = pd.to_numeric(  # text -> number
        trip_endpoints[col], errors="coerce"
    )

print("Trips with coordinates:", len(trip_endpoints))  # number of trips
print("Missing coordinates:",  # trips with a missing coordinate (should be 0)
      trip_endpoints[
          ["first_stop_lat", "first_stop_lon",
           "last_stop_lat", "last_stop_lon"]
      ].isna().any(axis=1).sum())

display(trip_endpoints[  # show the first 10 trips
    ["route_short_name", "direction_id",
     "first_stop_name", "last_stop_name",
     "first_stop_lat", "first_stop_lon",
     "last_stop_lat", "last_stop_lon"]
].head(10))


Trips with coordinates: 3259
Missing coordinates: 0


,route_short_name,direction_id,first_stop_name,last_stop_name,first_stop_lat,first_stop_lon,last_stop_lat,last_stop_lon
0,1,0,"Utrecht, Vechtsebanen","Utrecht, IJsselsteinlaan",52.123306,5.087202,52.059633,5.120174
1,1,1,"Utrecht, IJsselsteinlaan","Utrecht, Vechtsebanen",52.059633,5.120174,52.123306,5.087202
2,1,0,"Utrecht, Vechtsebanen","Utrecht, IJsselsteinlaan",52.123306,5.087202,52.059633,5.120174
3,1,1,"Utrecht, IJsselsteinlaan","Utrecht, Vechtsebanen",52.059633,5.120174,52.123306,5.087202
4,1,0,"Utrecht, Vechtsebanen","Utrecht, IJsselsteinlaan",52.123306,5.087202,52.059633,5.120174
5,1,1,"Utrecht, IJsselsteinlaan","Utrecht, Vechtsebanen",52.059633,5.120174,52.123306,5.087202
6,1,0,"Utrecht, Vechtsebanen","Utrecht, IJsselsteinlaan",52.123306,5.087202,52.059633,5.120174
7,1,1,"Utrecht, IJsselsteinlaan","Utrecht, Vechtsebanen",52.059633,5.120174,52.123306,5.087202
8,1,0,"Utrecht, Vechtsebanen","Utrecht, IJsselsteinlaan",52.123306,5.087202,52.059633,5.120174
9,1,1,"Utrecht, IJsselsteinlaan","Utrecht, Vechtsebanen",52.059633,5.120174,52.123306,5.087202


In [ ]:

# Find the start/end stop combination per bus line
route_endpoints = (  # count trips per line and start/end stop combination
    trip_endpoints
    .groupby([
        "route_short_name",
        "first_stop_id",
        "last_stop_id"
    ])
    .size()  # number of trips per combination
    .reset_index(name="number_of_trips")  # back to a table
)

# Select the most frequent combination per bus line
route_endpoints = (  # keep the most frequent combination per line
    route_endpoints
    .sort_values("number_of_trips", ascending=False)  # most trips first
    .drop_duplicates("route_short_name")  # first (= most frequent) row per line
)

# Add stop names and coordinates
route_endpoints = route_endpoints.merge(  # add stop names and coordinates
    trip_endpoints[[
        "first_stop_id", "last_stop_id",
        "first_stop_name", "last_stop_name",
        "first_stop_lat", "first_stop_lon",
        "last_stop_lat", "last_stop_lon"
    ]].drop_duplicates(),
    on=["first_stop_id", "last_stop_id"],
    how="left"
)

print("Bus lines:", route_endpoints["route_short_name"].nunique())  # check: one row per line

display(route_endpoints.head())  # show the table


Bus lines: 44


,route_short_name,first_stop_id,last_stop_id,number_of_trips,first_stop_name,last_stop_name,first_stop_lat,first_stop_lon,last_stop_lat,last_stop_lon
0,3,3923772,3923773,142,"Utrecht, CS Centrumzijde (A2)","Utrecht, CS Centrumzijde (A1)",52.090234,5.110959,52.090136,5.111077
1,28,3924076,3924412,94,"Utrecht, P+R Science Park","Vleuten, Station Vleuten (B)",52.091846,5.178043,52.102280,5.008046
2,73,3925042,3924175,74,"Maarssenbroek, Station Maarssen (B)","Zeist, Busstation (B)",52.136257,5.031814,52.089453,5.249726
3,7,3924917,3924293,72,"Utrecht, Professor Bavinckstraat","Utrecht, P+R Westraven",52.119053,5.072717,52.055926,5.106972
4,77,3924547,3924180,72,"Nieuwegein, Nieuwegein City (C)","Bilthoven, Station Bilthoven (B)",52.029118,5.079424,52.130738,5.205259


In [ ]:


# Addresses of the bus depots
depots = {  # depot name -> address
    "Westraven": "Griffioenlaan 8, Utrecht, Netherlands",
    "Nieuwegein": "Remiseweg 7, Nieuwegein, Netherlands",
    "Lage Weide": "Protonweg 13, Utrecht, Netherlands"
}

results = []  # list that gets one dict per depot

for name, address in depots.items():  # loop over the depots

    response = requests.get(  # ask OpenStreetMap Nominatim for the coordinates
        "https://nominatim.openstreetmap.org/search",
        params={  # search text, JSON answer, best match only
            "q": address,
            "format": "json",
            "limit": 1
        },
        headers={  # Nominatim requires an identifying User-Agent
            "User-Agent": "UtrechtBusDepotResearch/1.0 (academic research)"
        },
        timeout=20  # give up after 20 seconds
    )
    response.raise_for_status()  # stop on an HTTP error
    data = response.json()  # answer as a Python list

    if data:  # a location was found
        results.append({  # store depot, address, latitude, longitude and matched address
            "depot": name,
            "address": address,
            "latitude": float(data[0]["lat"]),
            "longitude": float(data[0]["lon"]),
            "matched_location": data[0]["display_name"]
        })
    else:  # nothing found
        print("No location found:", name)  # print a check / summary line

    time.sleep(1.1)  # Nominatim allows at most 1 request per second

depot_locations = pd.DataFrame(results)  # list -> table

display(depot_locations)  # show the table


,depot,address,latitude,longitude,matched_location
0,Westraven,"Griffioenlaan 8, Utrecht, Netherlands",52.056075,5.105179,"U-OV Westraven, 8, Griffioenlaan, Bedrijvengeb..."
1,Nieuwegein,"Remiseweg 7, Nieuwegein, Netherlands",52.051076,5.102734,"Remiseweg, Huis de Geer, Nieuwegein, Utrecht, ..."
2,Lage Weide,"Protonweg 13, Utrecht, Netherlands",52.108936,5.070016,"13, Protonweg, Bedrijvengebied Lage Weide, Nie..."


In [ ]:
# Calculate driving distances using OSRM

endpoint_cols = [  # names of the end-point columns (for reference; not used below)
    "first_stop_id", "first_stop_lat", "first_stop_lon",
    "last_stop_id", "last_stop_lat", "last_stop_lon"
]

# Collect unique terminal stops
first = route_endpoints[  # first stops: ID and coordinates
    ["first_stop_id", "first_stop_lat", "first_stop_lon"]
].copy()
first.columns = ["stop_id", "lat", "lon"]  # same column names for first and last stops

last = route_endpoints[  # last stops: ID and coordinates
    ["last_stop_id", "last_stop_lat", "last_stop_lon"]
].copy()
last.columns = ["stop_id", "lat", "lon"]  # same column names

terminals = pd.concat([first, last]).drop_duplicates("stop_id")  # all unique end stops

depot_coords = list(zip(  # depot coordinates as (lon, lat) pairs, the order OSRM uses
    depot_locations["longitude"],
    depot_locations["latitude"]
))

distances = []  # list for the results

# Process stops in small batches
for start in range(0, len(terminals), 20):  # 20 stops per request (limit of the public OSRM server)

    batch = terminals.iloc[start:start + 20]  # the stops of this batch

    stop_coords = list(zip(batch["lon"], batch["lat"]))  # their (lon, lat) pairs
    coords = depot_coords + stop_coords  # depots first, then the stops

    coord_string = ";".join(  # 'lon,lat;lon,lat;...' as OSRM expects
        f"{lon},{lat}" for lon, lat in coords
    )

    n_depots = len(depot_coords)  # number of depots (3)
    depot_idx = ";".join(map(str, range(n_depots)))  # positions of the depots in the list: '0;1;2'
    stop_idx = ";".join(map(str, range(n_depots, len(coords))))  # positions of the stops

    url = (  # OSRM table service: road distances between many points
        "https://router.project-osrm.org/table/v1/driving/"
        + coord_string
    )

    # Depot -> stop
    outward = requests.get(  # road distances from each depot to each stop
        url,
        params={
            "sources": depot_idx,  # from the depots
            "destinations": stop_idx,  # to the stops
            "annotations": "distance"  # return distances (m), not travel times
        },
        timeout=60
    )
    outward.raise_for_status()  # stop on an HTTP error
    outward = outward.json()  # answer as a dictionary

    time.sleep(1.1)  # be polite to the public server

    # Stop -> depot
    inward = requests.get(  # road distances from each stop back to each depot
        url,
        params={
            "sources": stop_idx,  # from the stops
            "destinations": depot_idx,  # to the depots
            "annotations": "distance"  # distances (m)
        },
        timeout=60
    )
    inward.raise_for_status()  # stop on an HTTP error
    inward = inward.json()  # answer as a dictionary

    if outward["code"] != "Ok" or inward["code"] != "Ok":  # OSRM reports success with code 'Ok'
        raise RuntimeError("OSRM distance calculation failed")  # stop with an error message

    for j, stop_id in enumerate(batch["stop_id"]):  # every stop of this batch (j = its position)
        for i, depot in enumerate(depot_locations["depot"]):  # every depot (i = its position)
            distances.append({  # one row per stop-depot pair
                "stop_id": stop_id,
                "depot": depot,
                "depot_to_stop_km":  # metres -> km; NaN if no route was found
                    outward["distances"][i][j] / 1000
                    if outward["distances"][i][j] is not None
                    else np.nan,
                "stop_to_depot_km":  # metres -> km; NaN if no route was found
                    inward["distances"][j][i] / 1000
                    if inward["distances"][j][i] is not None
                    else np.nan
            })

    time.sleep(1.1)  # pause before the next batch

distance_matrix = pd.DataFrame(distances)  # table: stop, depot, km out, km back

display(distance_matrix.head())  # show the table


,stop_id,depot,depot_to_stop_km,stop_to_depot_km
0,3923772,Westraven,5.1895,5.0350
1,3923772,Nieuwegein,5.8611,5.6862
2,3923772,Lage Weide,9.3471,11.1613
3,3924076,Westraven,9.9654,10.1942
4,3924076,Nieuwegein,10.6370,10.8454


In [ ]:

# Calculate depot distances for both directions
results = []  # one dict per line-depot pair

for _, line in route_endpoints.iterrows():  # every bus line

    for depot in depot_locations["depot"]:  # every depot

        start = distance_matrix[  # distances between this depot and the first stop of the line
            (distance_matrix["stop_id"] == line["first_stop_id"]) &
            (distance_matrix["depot"] == depot)
        ].iloc[0]

        end = distance_matrix[  # distances between this depot and the last stop of the line
            (distance_matrix["stop_id"] == line["last_stop_id"]) &
            (distance_matrix["depot"] == depot)
        ].iloc[0]

        # Direction A -> B
        outward_km = (  # depot -> first stop, and last stop -> depot
            start["depot_to_stop_km"]
            + end["stop_to_depot_km"]
        )

        # Direction B -> A
        return_km = (  # depot -> last stop, and first stop -> depot
            end["depot_to_stop_km"]
            + start["stop_to_depot_km"]
        )

        results.append({  # store both directions and their average
            "route_short_name": line["route_short_name"],
            "depot": depot,
            "outward_km": outward_km,
            "return_km": return_km,
            "average_empty_km": (outward_km + return_km) / 2  # average empty km per day (one pull-out + one pull-in)
        })

# Create DataFrame
depot_distances = pd.DataFrame(results)  # list -> table

# Select depot with shortest average empty distance
final_assignment = (  # per line the depot with the fewest empty km
    depot_distances
    .sort_values("average_empty_km")  # smallest first
    .drop_duplicates("route_short_name")  # keep the first (= nearest) depot per line
    .reset_index(drop=True)  # clean row numbers
)

display(final_assignment)  # show the table


,route_short_name,depot,outward_km,return_km,average_empty_km
0,3,Westraven,10.2245,10.2245,10.22450
1,2,Westraven,10.2245,10.2245,10.22450
2,9,Westraven,11.3546,9.3392,10.34690
3,565,Nieuwegein,10.8284,10.7778,10.80310
4,12,Lage Weide,10.3814,12.3741,11.37775
5,8,Westraven,11.6237,11.9509,11.78730
6,18,Westraven,11.8089,12.0638,11.93635
7,128,Lage Weide,12.3421,12.3359,12.33900
8,19,Westraven,12.3294,13.0956,12.71250
9,16,Westraven,12.3482,13.0768,12.71250


In [ ]:

# Add depot assignment to the bus line dataset
bus_lines_final = bus_lines.merge(  # add depot and empty km to every line
    final_assignment[
        ["route_short_name", "depot",
         "outward_km", "return_km", "average_empty_km"]
    ],
    on="route_short_name",
    how="left",
    validate="one_to_one"  # one depot per line (checked by pandas)
)

# Check results
print("Total bus lines:", len(bus_lines_final))
print("Missing depot assignments:",  # should be 0
      bus_lines_final["depot"].isna().sum())

display(bus_lines_final.head(15))  # show the first 15 lines


Total bus lines: 44
Missing depot assignments: 0


,route_short_name,Aantal bussen,Spitsfrequentie (p.u.),Modellen,shortest_km,average_km,longest_km,shortest_time,average_time,longest_time,aantal_ritten,average_speed,depot,outward_km,return_km,average_empty_km
0,1,8,5.0,7900 EA; Urbino 18 Electric,12.445404,12.841189,13.396250,37.0,47.521127,53.0,142,16.213239,Lage Weide,22.3361,15.6523,18.99420
1,2,1,2.0,E-way 10.7,5.048149,5.048149,5.048149,19.0,22.346154,23.0,26,13.554411,Westraven,10.2245,10.2245,10.22450
2,3,13,5.0,GX437 ELEC,5.083383,11.913150,12.188609,16.0,45.594595,52.0,148,15.677055,Westraven,10.2245,10.2245,10.22450
3,5,15,4.0,7900 EA; Citea LF-122 Electric; U12 LE; Urbino...,5.516617,15.077154,19.506721,19.0,49.551020,66.0,98,18.256521,Lage Weide,14.7421,14.5990,14.67055
4,6,9,4.0,7900 EA; Citea LF-122 Electric; U12 LE; Urbino...,6.767892,12.990394,15.372934,23.0,43.492308,56.0,130,17.920954,Lage Weide,12.5248,14.4309,13.47785
5,7,8,5.0,GX437 ELEC,11.459491,11.464402,11.469313,38.0,45.347222,50.0,144,15.168826,Westraven,16.8157,9.6140,13.21485
6,8,8,5.0,GX437 ELEC,11.117236,11.132362,11.147069,34.0,40.542254,44.0,142,16.475200,Westraven,11.6237,11.9509,11.78730
7,9,4,8.0,7900 EA; Urbino 18 Electric,3.305388,3.356049,3.409633,11.0,11.906542,14.0,107,16.911959,Westraven,11.3546,9.3392,10.34690
8,10,2,2.0,Citea LF-122 Electric; U12 LE; Urbino 9 Electric,9.240024,9.259223,9.278422,23.0,26.269231,29.0,52,21.148446,Westraven,14.5551,12.2087,13.38190
9,11,4,2.0,Citea LF-122 Electric; U12 LE; Urbino 9 Electric,5.333964,10.740238,10.953792,13.0,32.230769,42.0,52,19.993761,Lage Weide,14.7970,14.5863,14.69165


In [ ]:
# Check final depot assignments

print("Total bus lines:", len(bus_lines_final))  # number of lines

print(  # print a check / summary line
    "Missing depot assignments:",  # should be 0
    bus_lines_final["depot"].isna().sum()
)

# Display final depot assignments
display(  # line, buses, models, depot, km, trips and empty km
    bus_lines_final[
        [
            "route_short_name",
            "Aantal bussen",
            "Modellen",
            "depot",
            "average_km",
            "aantal_ritten",
            "average_empty_km"
        ]
    ]
)


Total bus lines: 44
Missing depot assignments: 0


,route_short_name,Aantal bussen,Modellen,depot,average_km,aantal_ritten,average_empty_km
0,1,8,7900 EA; Urbino 18 Electric,Lage Weide,12.841189,142,18.99420
1,2,1,E-way 10.7,Westraven,5.048149,26,10.22450
2,3,13,GX437 ELEC,Westraven,11.913150,148,10.22450
3,5,15,7900 EA; Citea LF-122 Electric; U12 LE; Urbino...,Lage Weide,15.077154,98,14.67055
4,6,9,7900 EA; Citea LF-122 Electric; U12 LE; Urbino...,Lage Weide,12.990394,130,13.47785
5,7,8,GX437 ELEC,Westraven,11.464402,144,13.21485
6,8,8,GX437 ELEC,Westraven,11.132362,142,11.78730
7,9,4,7900 EA; Urbino 18 Electric,Westraven,3.356049,107,10.34690
8,10,2,Citea LF-122 Electric; U12 LE; Urbino 9 Electric,Westraven,9.259223,52,13.38190
9,11,4,Citea LF-122 Electric; U12 LE; Urbino 9 Electric,Lage Weide,10.740238,52,14.69165


Part 2: Assignment of specific number of busses per model to the different buslines

In [ ]:
# PART 1: Loading and cleaning the datasets

# Normalizing model names across both datasets
# Function: write bus model names the same way in the line table and the fleet file
def normaliseer_model(model):
    model = str(model).strip()  # as text without spaces

    correcties = {  # known spelling differences -> one name
        "U18LF": "U18 LF",
        "7900 EA (U-link)": "7900 EA",
        "7900 EA (Stads- en streeklijnen)": "7900 EA",
        "2.2": "2.2 LE"
    }

    return correcties.get(model, model)  # corrected name, or the original if there is no correction


# Use the bus line and depot assignments calculated in Python
df = bus_lines_final.copy()  # start from the line table with depots

# Check that the required columns are available
required_columns = [  # columns this part needs
    "route_short_name",
    "Aantal bussen",
    "Modellen",
    "depot"
]

missing_columns = [  # required columns that are not in the table
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:  # stop if a column is missing
    raise ValueError(  # stop with an error message
        f"Missing columns: {missing_columns}"
    )

print("Bus lines loaded:", len(df))  # check
print("Total required buses:", df["Aantal bussen"].sum())  # check
# Cleaning and normalizing model names
df["Modellen"] = df["Modellen"].fillna("").apply(  # clean every model name in the list of each line
    lambda x: "; ".join(
        normaliseer_model(m)
        for m in str(x).split(";")
        if m.strip()
    )
)

# Number of eligible models per bus line
df["Aantal modellen"] = df["Modellen"].apply(  # how many different models can run on the line
    lambda x: len([m for m in x.split(";") if m.strip()])
)


# PART 2: Assigning buses to lines with only one eligible model

df_vast = df[  # lines with exactly one possible model and at least one bus
    (df["Aantal modellen"] == 1) &
    (df["Aantal bussen"] > 0)
].copy()

df_vast["Toegewezen model"] = df_vast["Modellen"]  # these lines get that model

display(df_vast[  # show the fixed assignments
    ["route_short_name", "depot",
     "Aantal bussen", "Toegewezen model"]
])

# Total assigned buses per model
toegewezen = (  # buses already used per model
    df_vast.groupby("Toegewezen model")["Aantal bussen"]
    .sum()
    .reset_index()
    .rename(columns={
        "Toegewezen model": "Model",
        "Aantal bussen": "Toegewezen"
    })
)


# PART 3: Calculating the remaining fleet

# Load fleet data from the central Excel file
df_vloot = pd.read_excel(  # fleet file: buses per model
    "Distribution of the busses in Utrecht.xlsx",
    sheet_name="Blad1",
    header=1
)

# Check total fleet excluding the Excel summary row
fleet_without_total = df_vloot[  # fleet rows without the 'Total' row of the Excel sheet
    ~df_vloot["Company"]
    .astype(str)
    .str.strip()
    .str.casefold()
    .isin(["total", "totaal"])
]

print(  # print a check / summary line
    "Total available buses:",  # sum of the fleet (check)
    pd.to_numeric(
        fleet_without_total["adjusted number"],
        errors="coerce"
    ).sum()
)

# Removing unnecessary whitespace from column names
df_vloot.columns = df_vloot.columns.str.strip()  # column names without spaces

# Checking whether the adjusted fleet column exists
if "adjusted number" not in df_vloot.columns:  # stop if the fleet column is missing
    raise ValueError(  # stop with an error message
        "Column 'adjusted number' not found. "
        "Check the Excel file and column name."
    )

# Selecting relevant columns
df_vloot = df_vloot[  # keep maker, model and number
    ["Company", "type", "adjusted number"]
].copy()

# Keeping rows with valid fleet quantities
df_vloot["adjusted number"] = pd.to_numeric(  # number of buses as a number (text -> NaN)
    df_vloot["adjusted number"],
    errors="coerce"
)

df_vloot = df_vloot.dropna(  # drop rows without maker, model or number
    subset=["Company", "type", "adjusted number"]
).copy()

# Checking that fleet quantities are non-negative integers
if (
    (df_vloot["adjusted number"] < 0).any()  # negative numbers
    or (df_vloot["adjusted number"] % 1 != 0).any()  # or numbers with decimals
):
    raise ValueError(  # stop with an error message
        "Fleet quantities must be non-negative whole numbers."
    )

df_vloot["adjusted number"] = (  # whole numbers
    df_vloot["adjusted number"].astype(int)
)

# Normalizing fleet model names
df_vloot["Model"] = (  # uniform model names
    df_vloot["type"].apply(normaliseer_model)
)

# Calculating total available buses per manufacturer and model
df_vloot = (  # total buses per maker and model (Dutch column names)
    df_vloot.groupby(
        ["Company", "Model"], as_index=False
    )["adjusted number"]
    .sum()
    .rename(columns={
        "Company": "Fabrikant",
        "adjusted number": "Totaal beschikbaar"
    })
)

# Checking for models assigned in Part 2 but missing from fleet
ontbrekende_modellen = (  # models used above that are not in the fleet file
    set(toegewezen["Model"]) - set(df_vloot["Model"])
)

if ontbrekende_modellen:  # stop if there are any
    raise ValueError(  # stop with an error message
        f"Assigned models missing from fleet: {ontbrekende_modellen}"
    )

# Checking that each model has only one manufacturer
if df_vloot["Model"].duplicated().any():  # a model may belong to one maker only
    raise ValueError(  # stop with an error message
        "Some model names occur under multiple manufacturers."
    )

# Combining fleet data with assigned buses
df_resterend = df_vloot.merge(  # fleet + buses already assigned, per model
    toegewezen,
    on="Model",
    how="left",
    validate="one_to_one"
)

# Models without assignments receive zero
df_resterend["Toegewezen"] = (  # no assignment -> 0
    df_resterend["Toegewezen"].fillna(0).astype(int)
)

# Calculating remaining buses per model
df_resterend["Resterend"] = (  # buses still free = available - assigned
    df_resterend["Totaal beschikbaar"]
    - df_resterend["Toegewezen"]
)

# Checking for negative remaining quantities
if (df_resterend["Resterend"] < 0).any():  # more assigned than available?
    print("WARNING: More buses assigned than available!")  # print a check / summary line
    display(df_resterend[df_resterend["Resterend"] < 0])  # show the table

# Adding totals
totalen = pd.DataFrame([{  # extra row with the totals
    "Fabrikant": "TOTAAL",
    "Model": "",
    "Totaal beschikbaar": df_resterend["Totaal beschikbaar"].sum(),
    "Toegewezen": df_resterend["Toegewezen"].sum(),
    "Resterend": df_resterend["Resterend"].sum()
}])

df_resterend = pd.concat(  # add the total row at the bottom
    [df_resterend, totalen],
    ignore_index=True
)

# Displaying the remaining fleet
display(df_resterend)  # show the remaining fleet


Bus lines loaded: 44
Total required buses: 311


,route_short_name,depot,Aantal bussen,Toegewezen model
1,2,Westraven,1,E-way 10.7
2,3,Westraven,13,GX437 ELEC
5,7,Westraven,8,GX437 ELEC
6,8,Westraven,8,GX437 ELEC
16,28,Westraven,16,Urbino 24 Electric
22,43,Westraven,2,Urbino 9 Electric
23,44,Nieuwegein,2,Urbino 9 Electric
27,57,Westraven,2,Urbino 9 Electric
34,81,Nieuwegein,2,Urbino 9 Electric
36,85,Nieuwegein,13,Urbino 15 Electric


Total available buses: 308.0


,Fabrikant,Model,Totaal beschikbaar,Toegewezen,Resterend
0,Ebusco,2.2 LE,20,0,20
1,Heuliez,GX437 ELEC,35,29,6
2,Iveco,E-way 10.7,3,1,2
3,Solaris,Urbino 15 Electric,16,13,3
4,Solaris,Urbino 18 Electric,52,0,52
5,Solaris,Urbino 24 Electric,25,16,9
6,Solaris,Urbino 9 Electric,13,10,3
7,VDL,Citea LF-122 Electric,17,0,17
8,Volvo,7900 EA,52,0,52
9,Yutong,U12 LE,27,0,27


In [ ]:

# STEP 3: Proportional assignment of remaining buses

# 1. Prepare fleet availability
fleet = df_resterend[  # fleet table without the TOTAL row
    ~df_resterend["Fabrikant"]
    .astype(str)
    .str.strip()
    .str.upper()
    .isin(["TOTAL", "TOTAAL"])
].copy()

fleet["Model"] = fleet["Model"].apply(normaliseer_model)  # uniform model names

fleet = fleet.groupby(  # one row per maker and model
    ["Fabrikant", "Model"], as_index=False
)[["Totaal beschikbaar", "Toegewezen", "Resterend"]].sum()

if fleet["Model"].duplicated().any():  # stop if a model name occurs twice
    raise ValueError("Duplicate bus model names in fleet.")  # stop with an error message

if (fleet["Resterend"] < 0).any():  # stop if the fixed lines use more buses than exist
    raise ValueError("Fixed assignments exceed available fleet.")  # stop with an error message

capacity = dict(zip(  # model -> total buses of that model
    fleet["Model"],
    fleet["Totaal beschikbaar"].astype(int)
))

remaining_capacity = dict(zip(  # model -> buses still free after the fixed lines
    fleet["Model"],
    fleet["Resterend"].astype(int)
))

manufacturers = dict(zip(  # model -> maker
    fleet["Model"],
    fleet["Fabrikant"]
))


# 2. Prepare bus lines
lines = df.copy()  # copy of the line table

lines["route_short_name"] = (  # uniform line numbers
    lines["route_short_name"]
    .apply(normalise_line_number)
)

lines["Aantal bussen"] = pd.to_numeric(  # buses needed as whole numbers (error if not numeric)
    lines["Aantal bussen"],
    errors="raise"
).astype(int)

lines["Eligible models"] = lines["Modellen"].fillna("").apply(  # list of possible models per line (no duplicates)
    lambda value: list(dict.fromkeys(
        normaliseer_model(model)
        for model in str(value).split(";")
        if model.strip()
    ))
)

lines["Number of models"] = (  # length of that list
    lines["Eligible models"].apply(len)
)


# 3. Exclude lines without eligible models
excluded_lines = lines[  # lines without any possible model
    lines["Number of models"] == 0
].copy()

lines = lines[  # continue with lines that have at least one model
    lines["Number of models"] > 0
].copy()

print("Excluded bus lines:")  # show the excluded lines
display(  # show the table
    excluded_lines[
        ["route_short_name", "Aantal bussen", "depot"]
    ]
)


# 4. Verify eligible models
eligible_models = {  # all models that appear on any line
    model
    for models in lines["Eligible models"]
    for model in models
}

unknown_models = eligible_models - set(capacity)  # models on lines that are not in the fleet

if unknown_models:  # stop if there are any
    raise ValueError(  # stop with an error message
        f"Models missing from fleet: {sorted(unknown_models)}"
    )


# 5. Preserve fixed assignments
results = []  # final list: one dict per line-model assignment

fixed_lines = lines[  # lines with exactly one possible model
    lines["Number of models"] == 1
].copy()

for _, line in fixed_lines.iterrows():  # every fixed line

    amount = int(line["Aantal bussen"])  # buses needed

    if amount <= 0:  # no buses needed -> skip
        continue  # skip to the next item

    model = line["Eligible models"][0]  # the only possible model

    results.append({  # store line, depot, maker, model, number and method 'fixed'
        "Lijn": line["route_short_name"],
        "Remise": line["depot"],
        "Fabrikant": manufacturers[model],
        "Model": model,
        "Aantal": amount,
        "Methode": "Vaste toewijzing"
    })


# 6. Prepare flexible lines
flexible = lines[  # lines with 2+ possible models and at least one bus
    (lines["Number of models"] > 1) &
    (lines["Aantal bussen"] > 0)
].copy().reset_index(drop=True)

models = sorted(capacity)  # all models, sorted

# Create one decision variable for every eligible
# combination of bus line and model
edges = [  # list of (line row, model): one variable x per possible pair
    (i, model)
    for i, line in flexible.iterrows()
    for model in line["Eligible models"]
]

n_edges = len(edges)  # number of line-model pairs
n_lines = len(flexible)  # number of flexible lines

# Variable layout:
# x: integer buses assigned to each line/model
# s: integer shortages per line
# d: absolute deviation from proportional target

n_vars = 2 * n_edges + n_lines  # x per pair + s per line + d per pair

x_offset = 0  # x variables start at position 0
s_offset = n_edges  # s (shortage) variables come after the x's
d_offset = n_edges + n_lines  # d (deviation) variables come last

objective = np.zeros(n_vars)  # cost per variable (start at 0)

lower_bounds = np.zeros(n_vars)  # all variables >= 0
upper_bounds = np.full(n_vars, np.inf)  # no upper limit here (capacity is a constraint)

integrality = np.zeros(n_vars, dtype=int)  # 0 = continuous variable

# Bus counts and shortages must be integers
integrality[:n_edges + n_lines] = 1  # x and s must be whole numbers (whole buses)

# Shortages have the highest penalty
objective[  # cost 10,000 per missing bus: avoid shortages first
    s_offset:s_offset + n_lines
] = 10000

# Proportional deviations have a smaller penalty
objective[d_offset:] = 1  # cost 1 per bus of deviation from the proportional target


# 7. Calculate proportional targets
targets = {}  # (line row, model) -> proportional target number of buses

for i, line in flexible.iterrows():  # every flexible line

    eligible = line["Eligible models"]  # its possible models

    total_available = sum(  # free buses of all those models together
        remaining_capacity[model]
        for model in eligible
    )

    for model in eligible:  # every possible model

        if total_available > 0:  # share of this model = its free buses / all free buses
            target = (  # target = buses needed x share
                int(line["Aantal bussen"])
                * remaining_capacity[model]
                / total_available
            )
        else:  # no free buses at all
            target = 0  # -> target 0

        targets[(i, model)] = target  # store the target


# 8. Build optimisation constraints
# Each line must receive its required buses,
# or the difference is recorded as a shortage.

n_constraints = (  # one per line + one per model + two per pair
    n_lines
    + len(models)
    + 2 * n_edges
)

A = lil_matrix(  # constraint matrix: rows = constraints, columns = variables
    (n_constraints, n_vars),
    dtype=float
)

constraint_lower = np.full(  # lower limit of each constraint (start: none)
    n_constraints, -np.inf
)

constraint_upper = np.full(  # upper limit of each constraint (start: none)
    n_constraints, np.inf
)

row = 0  # current constraint row

# Demand constraints per line
for i, line in flexible.iterrows():  # one demand constraint per line

    for j, (line_i, model) in enumerate(edges):  # all pairs ...
        if line_i == i:  # ... of this line
            A[row, j] = 1  # + x(line, model)

    A[row, s_offset + i] = 1  # + shortage of this line

    required = int(line["Aantal bussen"])  # buses needed

    constraint_lower[row] = required  # sum of x + shortage = buses needed
    constraint_upper[row] = required  # (equality: lower = upper)

    row += 1  # next constraint


# Fleet availability constraints per model
for model in models:  # one capacity constraint per model

    for j, (line_i, edge_model) in enumerate(edges):  # all pairs ...
        if edge_model == model:  # ... with this model
            A[row, j] = 1  # + x(line, model)

    constraint_upper[row] = remaining_capacity[model]  # sum of x <= free buses of this model

    row += 1  # next constraint


# Absolute proportional deviation constraints
for j, (i, model) in enumerate(edges):  # two constraints per pair, together: d >= |x - target|

    target = targets[(i, model)]  # proportional target of this pair

    # x - d <= target
    A[row, j] = 1  # + x
    A[row, d_offset + j] = -1  # - d
    constraint_upper[row] = target  # <= target
    row += 1  # next constraint row

    # -x - d <= -target
    A[row, j] = -1  # - x
    A[row, d_offset + j] = -1  # - d
    constraint_upper[row] = -target  # <= -target
    row += 1  # next constraint row


# 9. Run global optimisation
solution = milp(  # solve the mixed-integer linear programme
    c=objective,  # costs to minimise
    integrality=integrality,  # which variables are whole numbers
    bounds=Bounds(lower_bounds, upper_bounds),  # variable limits
    constraints=LinearConstraint(  # lower <= A x <= upper
        A.tocsr(),  # matrix in a fast format for the solver
        constraint_lower,
        constraint_upper
    ),
    options={
        "time_limit": 120,  # stop after 120 seconds
        "mip_rel_gap": 0.0  # require the proven optimum
    }
)

if solution.x is None or solution.status != 0:  # stop if no optimal solution was found
    raise RuntimeError(  # stop with an error message
        f"Optimisation did not finish successfully: "
        f"{solution.message}"
    )


# 10. Store optimised assignments
for j, (i, model) in enumerate(edges):  # every line-model pair

    amount = int(round(solution.x[j]))  # optimal number of buses (whole number)

    if amount <= 0:  # no buses -> skip
        continue  # skip to the next item

    line = flexible.iloc[i]  # the line of this pair

    results.append({  # store with method 'global optimisation'
        "Lijn": line["route_short_name"],
        "Remise": line["depot"],
        "Fabrikant": manufacturers[model],
        "Model": model,
        "Aantal": amount,
        "Methode": "Globale optimalisatie"
    })


# 11. Store unavoidable shortages
shortages = []  # lines that cannot get enough buses

for i, line in flexible.iterrows():  # every flexible line

    shortage = int(round(  # value of its shortage variable
        solution.x[s_offset + i]
    ))

    if shortage > 0:  # store only real shortages
        shortages.append({  # add one row to the list
            "Lijn": line["route_short_name"],
            "Niet toegewezen": shortage
        })


# 12. Create result tables
df_assignment = pd.DataFrame(  # all assignments as a table
    results,
    columns=[
        "Lijn", "Remise", "Fabrikant",
        "Model", "Aantal", "Methode"
    ]
)

df_shortages = pd.DataFrame(  # all shortages as a table
    shortages,
    columns=["Lijn", "Niet toegewezen"]
)


# 13. Validate model availability
assigned_by_model = (  # buses assigned per model
    df_assignment.groupby("Model")["Aantal"].sum()
)

for model, amount in assigned_by_model.items():  # check every model ...

    if amount > capacity[model]:  # ... never more than the fleet has
        raise ValueError(  # stop with an error message
            f"Fleet exceeded for {model}: "
            f"{amount} > {capacity[model]}"
        )


# 14. Validate bus line demand
required_by_line = (  # buses needed per line
    lines.set_index("route_short_name")["Aantal bussen"]
)

assigned_by_line = (  # buses assigned per line
    df_assignment.groupby("Lijn")["Aantal"].sum()
)

shortage_by_line = (  # shortage per line
    df_shortages.groupby("Lijn")["Niet toegewezen"].sum()
)

for line, required in required_by_line.items():  # check every line ...

    assigned = int(assigned_by_line.get(line, 0))  # assigned (0 if none)
    shortage = int(shortage_by_line.get(line, 0))  # shortage (0 if none)

    if assigned + shortage != required:  # ... assigned + shortage must equal the demand
        raise ValueError(  # stop with an error message
            f"Demand mismatch on line {line}"
        )


# 15. Final fleet overview
df_fleet_final = fleet[  # fleet per model
    ["Fabrikant", "Model", "Totaal beschikbaar"]
].copy()

df_fleet_final["Toegewezen"] = (  # + buses assigned (0 if none)
    df_fleet_final["Model"]
    .map(assigned_by_model)
    .fillna(0)
    .astype(int)
)

df_fleet_final["Resterend"] = (  # + buses left over
    df_fleet_final["Totaal beschikbaar"]
    - df_fleet_final["Toegewezen"]
)


# 16. Final summary
required_total = int(lines["Aantal bussen"].sum())  # buses needed in total
assigned_total = int(df_assignment["Aantal"].sum())  # buses assigned in total
shortage_total = int(df_shortages["Niet toegewezen"].sum())  # buses missing in total
fleet_total = int(fleet["Totaal beschikbaar"].sum())  # buses in the fleet
unused_total = int(df_fleet_final["Resterend"].sum())  # buses not used

# Print the summary and show the three result tables
print("\nOPTIMISATION RESULTS")
print("--------------------")  # print a check / summary line
print("Bus lines:", len(lines))  # print a check / summary line
print("Required buses:", required_total)  # print a check / summary line
print("Assigned buses:", assigned_total)  # print a check / summary line
print("Shortage:", shortage_total)  # print a check / summary line
print("Available fleet:", fleet_total)  # print a check / summary line
print("Unused buses:", unused_total)  # print a check / summary line

print("\nASSIGNMENTS")  # print a check / summary line
display(df_assignment)  # show the table

print("\nFLEET OVERVIEW")  # print a check / summary line
display(df_fleet_final)  # show the table

print("\nSHORTAGES")  # print a check / summary line
display(df_shortages)  # show the table


Excluded bus lines:


,route_short_name,Aantal bussen,depot
39,122,2,Lage Weide
41,158,1,Westraven
43,565,3,Nieuwegein



OPTIMISATION RESULTS
--------------------
Bus lines: 41
Required buses: 305
Assigned buses: 293
Shortage: 12
Available fleet: 308
Unused buses: 15

ASSIGNMENTS


,Lijn,Remise,Fabrikant,Model,Aantal,Methode
0,2,Westraven,Iveco,E-way 10.7,1,Vaste toewijzing
1,3,Westraven,Heuliez,GX437 ELEC,13,Vaste toewijzing
2,7,Westraven,Heuliez,GX437 ELEC,8,Vaste toewijzing
3,8,Westraven,Heuliez,GX437 ELEC,8,Vaste toewijzing
4,28,Westraven,Solaris,Urbino 24 Electric,16,Vaste toewijzing
5,43,Westraven,Solaris,Urbino 9 Electric,2,Vaste toewijzing
6,44,Nieuwegein,Solaris,Urbino 9 Electric,2,Vaste toewijzing
7,57,Westraven,Solaris,Urbino 9 Electric,2,Vaste toewijzing
8,81,Nieuwegein,Solaris,Urbino 9 Electric,2,Vaste toewijzing
9,85,Nieuwegein,Solaris,Urbino 15 Electric,13,Vaste toewijzing



FLEET OVERVIEW


,Fabrikant,Model,Totaal beschikbaar,Toegewezen,Resterend
0,Ebusco,2.2 LE,20,20,0
1,Heuliez,GX437 ELEC,35,29,6
2,Iveco,E-way 10.7,3,3,0
3,Solaris,Urbino 15 Electric,16,16,0
4,Solaris,Urbino 18 Electric,52,52,0
5,Solaris,Urbino 24 Electric,25,16,9
6,Solaris,Urbino 9 Electric,13,13,0
7,VDL,Citea LF-122 Electric,17,17,0
8,Volvo,7900 EA,52,52,0
9,Yutong,U12 LE,27,27,0



SHORTAGES


,Lijn,Niet toegewezen
0,6,4
1,55,1
2,66,1
3,84,6


In [ ]:
pd.set_option("display.max_rows", None)  # show all rows of a table
pd.set_option("display.max_columns", None)  # show all columns

display(df_assignment)  # full assignment table

,Lijn,Remise,Fabrikant,Model,Aantal,Methode
0,2,Westraven,Iveco,E-way 10.7,1,Vaste toewijzing
1,3,Westraven,Heuliez,GX437 ELEC,13,Vaste toewijzing
2,7,Westraven,Heuliez,GX437 ELEC,8,Vaste toewijzing
3,8,Westraven,Heuliez,GX437 ELEC,8,Vaste toewijzing
4,28,Westraven,Solaris,Urbino 24 Electric,16,Vaste toewijzing
5,43,Westraven,Solaris,Urbino 9 Electric,2,Vaste toewijzing
6,44,Nieuwegein,Solaris,Urbino 9 Electric,2,Vaste toewijzing
7,57,Westraven,Solaris,Urbino 9 Electric,2,Vaste toewijzing
8,81,Nieuwegein,Solaris,Urbino 9 Electric,2,Vaste toewijzing
9,85,Nieuwegein,Solaris,Urbino 15 Electric,13,Vaste toewijzing


In [ ]:

# STEP 4: Validate the final bus model assignments

# Check overall fleet allocation
required_buses = int(lines["Aantal bussen"].sum())  # buses needed
assigned_buses = int(df_assignment["Aantal"].sum())  # buses assigned
missing_buses = int(df_shortages["Niet toegewezen"].sum())  # buses missing
available_buses = int(df_fleet_final["Totaal beschikbaar"].sum())  # buses in the fleet

print("FINAL BUS MODEL ASSIGNMENT")  # print a check / summary line
print("--------------------------")  # print a check / summary line
print("Bus lines:", len(lines))  # print a check / summary line
print("Required buses:", required_buses)  # print a check / summary line
print("Assigned buses:", assigned_buses)  # print a check / summary line
print("Bus shortage:", missing_buses)  # print a check / summary line
print("Available fleet:", available_buses)  # print a check / summary line
print("Unused buses:", available_buses - assigned_buses)  # fleet minus assigned

# Verify that the total demand is accounted for
assert assigned_buses + missing_buses == required_buses, (  # stop if buses got lost in the bookkeeping
    "Assigned buses and shortages do not match demand."
)

# Verify that fleet capacity is not exceeded
assert (df_fleet_final["Resterend"] >= 0).all(), (  # stop if a model is over-assigned
    "Bus model availability has been exceeded."
)

# Save the final assignment for subsequent calculations
df_assignment_final = df_assignment.copy()  # fixed copy used in the next steps

print("\nAssignment validated successfully.")  # print a check / summary line


FINAL BUS MODEL ASSIGNMENT
--------------------------
Bus lines: 41
Required buses: 305
Assigned buses: 293
Bus shortage: 12
Available fleet: 308
Unused buses: 15

Assignment validated successfully.


In [ ]:

# STEP 5: Calculate daily distance and energy consumption per bus

# 1. Load technical bus specifications from the central Excel file
specs = pd.read_excel(  # fleet file again, now for consumption and battery size
    "Distribution of the busses in Utrecht.xlsx",
    sheet_name="Blad1",
    header=1
)

specs.columns = specs.columns.str.strip()  # column names without spaces

specs = specs[  # maker, model, kWh/km (two sources) and battery capacity
    [
        "Company",
        "type",
        "Average independent source",
        "Average manufacturer",
        "Max. capacity (kWh)"
    ]
].copy()

specs = specs.dropna(subset=["type"]).copy()  # drop rows without a model

specs["Model"] = specs["type"].apply(normaliseer_model)  # uniform model names

# Prefer independent consumption data; use manufacturer data if unavailable
specs["Verbruik"] = pd.to_numeric(  # kWh/km: independent test value, else the manufacturer value
    specs["Average independent source"],
    errors="coerce"
).combine_first(
    pd.to_numeric(
        specs["Average manufacturer"],
        errors="coerce"
    )
)

specs["Capacity"] = pd.to_numeric(  # battery capacity in kWh
    specs["Max. capacity (kWh)"],
    errors="coerce"
)

# Check duplicate model specifications
if specs["Model"].duplicated().any():  # each model once
    raise ValueError("Duplicate model specifications found.")  # stop with an error message

specs = specs[  # keep model, consumption and capacity
    ["Model", "Verbruik", "Capacity"]
].copy()


# 2. Start with the final bus model assignments
df_verbruik = df_assignment_final.copy()  # one row per line-model assignment

df_verbruik["Lijn"] = (  # uniform line numbers (needed for the merge)
    df_verbruik["Lijn"].apply(normalise_line_number)
)


# 3. Get daily GTFS statistics and depot distances
lijngegevens = bus_lines_final[  # trips per day, average km and empty km per line
    [
        "route_short_name",
        "aantal_ritten",
        "average_km",
        "average_empty_km"
    ]
].copy()

lijngegevens["route_short_name"] = (  # uniform line numbers
    lijngegevens["route_short_name"].apply(normalise_line_number)
)

lijngegevens = lijngegevens.rename(columns={  # Dutch column names, the same as in df_verbruik
    "route_short_name": "Lijn",
    "aantal_ritten": "Aantal ritten",
    "average_km": "Gem km",
    "average_empty_km": "Lege km per bus"
})


# 4. Connect line statistics to bus assignments
df_verbruik = df_verbruik.merge(  # add trips and km of the line
    lijngegevens,
    on="Lijn",
    how="left",
    validate="many_to_one"
)


# 5. Add technical specifications
df_verbruik = df_verbruik.merge(  # add consumption and battery of the model
    specs,
    on="Model",
    how="left",
    validate="many_to_one"
)


# 6. Validate essential data
required = [  # columns that must be filled in
    "Aantal ritten",
    "Gem km",
    "Lege km per bus",
    "Verbruik"
]

missing = df_verbruik[  # rows where one of them is empty
    df_verbruik[required].isna().any(axis=1)
]

if not missing.empty:  # stop and show them
    display(missing)  # show the table
    raise ValueError(  # stop with an error message
        "Missing GTFS distances, depot distances or consumption data."
    )

# Battery capacity is needed later, but not for consumption
missing_capacity = df_verbruik[  # rows without a battery capacity (only a warning)
    df_verbruik["Capacity"].isna()
]

if not missing_capacity.empty:  # if any: print a warning and show the models
    print("WARNING: Missing battery capacity for:")  # print a check / summary line
    display(  # show the table
        missing_capacity[
            ["Model", "Fabrikant"]
        ].drop_duplicates()
    )


# 7. Calculate total assigned buses per line
df_verbruik["Totaal bussen lijn"] = (  # all buses of the line (all models together)
    df_verbruik.groupby("Lijn")["Aantal"].transform("sum")
)

if (df_verbruik["Totaal bussen lijn"] <= 0).any():  # a line must have buses
    raise ValueError("A bus line has no assigned buses.")  # stop with an error message


# 8. Calculate average trips per assigned bus
df_verbruik["Ritten per bus"] = (  # trips of the line spread evenly over its buses
    df_verbruik["Aantal ritten"]
    / df_verbruik["Totaal bussen lijn"]
)


# 9. Calculate daily service kilometres per bus
df_verbruik["Dienst km per bus"] = (  # trips per bus x km per trip
    df_verbruik["Ritten per bus"]
    * df_verbruik["Gem km"]
)


# 10. Add one daily depot departure and return
# average_empty_km already includes both journeys
df_verbruik["Totale km per bus"] = (  # service km + empty km to and from the depot
    df_verbruik["Dienst km per bus"]
    + df_verbruik["Lege km per bus"]
)


# 11. Calculate daily energy per bus
df_verbruik["Verbruik per bus (kWh)"] = (  # km per day x kWh per km
    df_verbruik["Totale km per bus"]
    * df_verbruik["Verbruik"]
)


# 12. Calculate total daily energy for each line/model
df_verbruik["Totaal verbruik (kWh)"] = (  # energy per bus x number of buses
    df_verbruik["Verbruik per bus (kWh)"]
    * df_verbruik["Aantal"]
)


# 13. Validate number of assigned buses
if df_verbruik["Aantal"].sum() != df_assignment_final["Aantal"].sum():  # the merges must not add or lose buses
    raise ValueError("Bus assignments were duplicated or lost.")  # stop with an error message


# 14. Display results
# Print the totals and show the detailed table
print("DAILY ENERGY ANALYSIS")
print("---------------------")  # print a check / summary line
print("Assigned buses:", df_verbruik["Aantal"].sum())  # print a check / summary line
print(  # print a check / summary line
    "Total daily energy (MWh):",
    round(df_verbruik["Totaal verbruik (kWh)"].sum() / 1000, 2)
)

display(  # show the table
    df_verbruik[
        [
            "Lijn",
            "Remise",
            "Fabrikant",
            "Model",
            "Aantal",
            "Aantal ritten",
            "Totaal bussen lijn",
            "Ritten per bus",
            "Dienst km per bus",
            "Lege km per bus",
            "Totale km per bus",
            "Verbruik",
            "Verbruik per bus (kWh)",
            "Totaal verbruik (kWh)",
            "Capacity"
        ]
    ]
)


DAILY ENERGY ANALYSIS
---------------------
Assigned buses: 293
Total daily energy (MWh): 78.5


,Lijn,Remise,Fabrikant,Model,Aantal,Aantal ritten,Totaal bussen lijn,Ritten per bus,Dienst km per bus,Lege km per bus,Totale km per bus,Verbruik,Verbruik per bus (kWh),Totaal verbruik (kWh),Capacity
0,2,Westraven,Iveco,E-way 10.7,1,26,1,26.000000,131.251878,10.22450,141.476378,0.920000,130.158268,130.158268,385.00
1,3,Westraven,Heuliez,GX437 ELEC,13,148,13,11.384615,135.626626,10.22450,145.851126,2.083333,303.856513,3950.134668,250.00
2,7,Westraven,Heuliez,GX437 ELEC,8,144,8,18.000000,206.359233,13.21485,219.574083,2.083333,457.446006,3659.568052,250.00
3,8,Westraven,Heuliez,GX437 ELEC,8,142,8,17.750000,197.599430,11.78730,209.386730,2.083333,436.222355,3489.778839,250.00
4,28,Westraven,Solaris,Urbino 24 Electric,16,186,16,11.625000,219.263869,21.91435,241.178219,2.707692,653.036409,10448.582537,704.00
5,43,Westraven,Solaris,Urbino 9 Electric,2,52,2,26.000000,280.649263,25.62845,306.277713,0.800000,245.022170,490.044341,350.00
6,44,Nieuwegein,Solaris,Urbino 9 Electric,2,49,2,24.500000,331.858040,22.22980,354.087840,0.800000,283.270272,566.540544,350.00
7,57,Westraven,Solaris,Urbino 9 Electric,2,28,2,14.000000,181.505983,30.51560,212.021583,0.800000,169.617266,339.234533,350.00
8,81,Nieuwegein,Solaris,Urbino 9 Electric,2,39,2,19.500000,274.834368,39.26870,314.103068,0.800000,251.282455,502.564909,350.00
9,85,Nieuwegein,Solaris,Urbino 15 Electric,13,91,13,7.000000,201.982697,30.24290,232.225597,1.880000,436.584123,5675.593596,470.00


In [ ]:

# STEP 5: Calculate daily distance and energy consumption per bus

# 1. Load technical bus specifications from the central Excel file
specs = pd.read_excel(  # fleet file again, now for consumption and battery size
    "Distribution of the busses in Utrecht.xlsx",
    sheet_name="Blad1",
    header=1
)

specs.columns = specs.columns.str.strip()  # column names without spaces

specs = specs[  # maker, model, kWh/km (two sources) and battery capacity
    [
        "Company",
        "type",
        "Average independent source",
        "Average manufacturer",
        "Max. capacity (kWh)"
    ]
].copy()

specs = specs.dropna(subset=["type"]).copy()  # drop rows without a model

specs["Model"] = specs["type"].apply(normaliseer_model)  # uniform model names

# Prefer independent consumption data; use manufacturer data if unavailable
specs["Verbruik"] = pd.to_numeric(  # kWh/km: independent test value, else the manufacturer value
    specs["Average independent source"],
    errors="coerce"
).combine_first(
    pd.to_numeric(
        specs["Average manufacturer"],
        errors="coerce"
    )
)

specs["Capacity"] = pd.to_numeric(  # battery capacity in kWh
    specs["Max. capacity (kWh)"],
    errors="coerce"
)

# Check duplicate model specifications
if specs["Model"].duplicated().any():  # each model once
    raise ValueError("Duplicate model specifications found.")  # stop with an error message

specs = specs[  # keep model, consumption and capacity
    ["Model", "Verbruik", "Capacity"]
].copy()


# 2. Start with the final bus model assignments
df_verbruik = df_assignment_final.copy()  # one row per line-model assignment

df_verbruik["Lijn"] = (  # uniform line numbers (needed for the merge)
    df_verbruik["Lijn"].apply(normalise_line_number)
)


# 3. Get daily GTFS statistics and depot distances
lijngegevens = bus_lines_final[  # trips per day, average km and empty km per line
    [
        "route_short_name",
        "aantal_ritten",
        "average_km",
        "average_empty_km"
    ]
].copy()

lijngegevens["route_short_name"] = (  # uniform line numbers
    lijngegevens["route_short_name"].apply(normalise_line_number)
)

lijngegevens = lijngegevens.rename(columns={  # Dutch column names, the same as in df_verbruik
    "route_short_name": "Lijn",
    "aantal_ritten": "Aantal ritten",
    "average_km": "Gem km",
    "average_empty_km": "Lege km per bus"
})


# 4. Connect line statistics to bus assignments
df_verbruik = df_verbruik.merge(  # add trips and km of the line
    lijngegevens,
    on="Lijn",
    how="left",
    validate="many_to_one"
)


# 5. Add technical specifications
df_verbruik = df_verbruik.merge(  # add consumption and battery of the model
    specs,
    on="Model",
    how="left",
    validate="many_to_one"
)


# 6. Validate essential data
required = [  # columns that must be filled in
    "Aantal ritten",
    "Gem km",
    "Lege km per bus",
    "Verbruik"
]

missing = df_verbruik[  # rows where one of them is empty
    df_verbruik[required].isna().any(axis=1)
]

if not missing.empty:  # stop and show them
    display(missing)  # show the table
    raise ValueError(  # stop with an error message
        "Missing GTFS distances, depot distances or consumption data."
    )

# Battery capacity is needed later, but not for consumption
missing_capacity = df_verbruik[  # rows without a battery capacity (only a warning)
    df_verbruik["Capacity"].isna()
]

if not missing_capacity.empty:  # if any: print a warning and show the models
    print("WARNING: Missing battery capacity for:")  # print a check / summary line
    display(  # show the table
        missing_capacity[
            ["Model", "Fabrikant"]
        ].drop_duplicates()
    )


# 7. Calculate total assigned buses per line
df_verbruik["Totaal bussen lijn"] = (  # all buses of the line (all models together)
    df_verbruik.groupby("Lijn")["Aantal"].transform("sum")
)

if (df_verbruik["Totaal bussen lijn"] <= 0).any():  # a line must have buses
    raise ValueError("A bus line has no assigned buses.")  # stop with an error message


# 8. Calculate average trips per assigned bus
df_verbruik["Ritten per bus"] = (  # trips of the line spread evenly over its buses
    df_verbruik["Aantal ritten"]
    / df_verbruik["Totaal bussen lijn"]
)


# 9. Calculate daily service kilometres per bus
df_verbruik["Dienst km per bus"] = (  # trips per bus x km per trip
    df_verbruik["Ritten per bus"]
    * df_verbruik["Gem km"]
)


# 10. Add one daily depot departure and return
# average_empty_km already includes both journeys
df_verbruik["Totale km per bus"] = (  # service km + empty km to and from the depot
    df_verbruik["Dienst km per bus"]
    + df_verbruik["Lege km per bus"]
)


# 11. Calculate daily energy per bus
df_verbruik["Verbruik per bus (kWh)"] = (  # km per day x kWh per km
    df_verbruik["Totale km per bus"]
    * df_verbruik["Verbruik"]
)


# 12. Calculate total daily energy for each line/model
df_verbruik["Totaal verbruik (kWh)"] = (  # energy per bus x number of buses
    df_verbruik["Verbruik per bus (kWh)"]
    * df_verbruik["Aantal"]
)


# 13. Validate number of assigned buses
if df_verbruik["Aantal"].sum() != df_assignment_final["Aantal"].sum():  # the merges must not add or lose buses
    raise ValueError("Bus assignments were duplicated or lost.")  # stop with an error message


# 14. Display results
# Print the totals and show the detailed table
print("DAILY ENERGY ANALYSIS")
print("---------------------")  # print a check / summary line
print("Assigned buses:", df_verbruik["Aantal"].sum())  # print a check / summary line
print(  # print a check / summary line
    "Total daily energy (MWh):",
    round(df_verbruik["Totaal verbruik (kWh)"].sum() / 1000, 2)
)

display(  # show the table
    df_verbruik[
        [
            "Lijn",
            "Remise",
            "Fabrikant",
            "Model",
            "Aantal",
            "Aantal ritten",
            "Totaal bussen lijn",
            "Ritten per bus",
            "Dienst km per bus",
            "Lege km per bus",
            "Totale km per bus",
            "Verbruik",
            "Verbruik per bus (kWh)",
            "Totaal verbruik (kWh)",
            "Capacity"
        ]
    ]
)


DAILY ENERGY ANALYSIS
---------------------
Assigned buses: 293
Total daily energy (MWh): 78.5


,Lijn,Remise,Fabrikant,Model,Aantal,Aantal ritten,Totaal bussen lijn,Ritten per bus,Dienst km per bus,Lege km per bus,Totale km per bus,Verbruik,Verbruik per bus (kWh),Totaal verbruik (kWh),Capacity
0,2,Westraven,Iveco,E-way 10.7,1,26,1,26.000000,131.251878,10.22450,141.476378,0.920000,130.158268,130.158268,385.00
1,3,Westraven,Heuliez,GX437 ELEC,13,148,13,11.384615,135.626626,10.22450,145.851126,2.083333,303.856513,3950.134668,250.00
2,7,Westraven,Heuliez,GX437 ELEC,8,144,8,18.000000,206.359233,13.21485,219.574083,2.083333,457.446006,3659.568052,250.00
3,8,Westraven,Heuliez,GX437 ELEC,8,142,8,17.750000,197.599430,11.78730,209.386730,2.083333,436.222355,3489.778839,250.00
4,28,Westraven,Solaris,Urbino 24 Electric,16,186,16,11.625000,219.263869,21.91435,241.178219,2.707692,653.036409,10448.582537,704.00
5,43,Westraven,Solaris,Urbino 9 Electric,2,52,2,26.000000,280.649263,25.62845,306.277713,0.800000,245.022170,490.044341,350.00
6,44,Nieuwegein,Solaris,Urbino 9 Electric,2,49,2,24.500000,331.858040,22.22980,354.087840,0.800000,283.270272,566.540544,350.00
7,57,Westraven,Solaris,Urbino 9 Electric,2,28,2,14.000000,181.505983,30.51560,212.021583,0.800000,169.617266,339.234533,350.00
8,81,Nieuwegein,Solaris,Urbino 9 Electric,2,39,2,19.500000,274.834368,39.26870,314.103068,0.800000,251.282455,502.564909,350.00
9,85,Nieuwegein,Solaris,Urbino 15 Electric,13,91,13,7.000000,201.982697,30.24290,232.225597,1.880000,436.584123,5675.593596,470.00


In [ ]:

# STEP 6: Calculate daily battery charging requirements

# 1. Start with the daily energy consumption results
df_battery = df_verbruik.copy()  # start from the energy table

# 2. Check battery capacities
df_battery["Capacity"] = pd.to_numeric(  # battery capacity as a number
    df_battery["Capacity"],
    errors="coerce"
)

if (  # stop if a capacity is ...
    df_battery["Capacity"].isna().any()  # missing capacity
    or (df_battery["Capacity"] <= 0).any()  # or zero/negative capacity
):
    raise ValueError(  # stop with an error message
        "Missing or invalid battery capacity."
    )

# 3. Calculate battery equivalents required per bus
df_battery["Battery equivalents"] = (  # daily energy / battery size (1.4 = 1.4 full batteries)
    df_battery["Verbruik per bus (kWh)"]
    / df_battery["Capacity"]
)

# 4. Calculate number of full battery charges required
df_battery["Required battery charges"] = np.ceil(  # rounded up: full charges needed per day
    df_battery["Battery equivalents"]
).astype(int)

# 5. Determine whether one overnight charge is sufficient
df_battery["Overnight charging sufficient"] = (  # True if one full battery covers the whole day
    df_battery["Verbruik per bus (kWh)"]
    <= df_battery["Capacity"]
)

# 6. Calculate additional energy required during operation
df_battery["Additional energy (kWh per bus)"] = (  # energy above one battery (0 if it fits)
    df_battery["Verbruik per bus (kWh)"]
    - df_battery["Capacity"]
).clip(lower=0)

# 7. Calculate additional charging sessions per bus
df_battery["Additional charging sessions"] = (  # charges needed besides the night charge
    df_battery["Required battery charges"] - 1
).clip(lower=0)

# 8. Calculate additional charging energy per line/model
df_battery["Total additional energy (kWh)"] = (  # extra energy x number of buses
    df_battery["Additional energy (kWh per bus)"]
    * df_battery["Aantal"]
)

# 9. Calculate number of buses requiring additional charging
df_battery["Buses requiring additional charging"] = np.where(  # all buses of the row if one battery is not enough, else 0
    df_battery["Overnight charging sufficient"],
    0,
    df_battery["Aantal"]
)

# 10. Display summary
# Print the totals
print("BATTERY CHARGING ANALYSIS")
print("-------------------------")  # print a check / summary line

print(  # print a check / summary line
    "Total assigned buses:",
    df_battery["Aantal"].sum()
)

print(  # print a check / summary line
    "Buses requiring additional charging:",
    df_battery["Buses requiring additional charging"].sum()
)

print(  # print a check / summary line
    "Additional charging energy (MWh):",
    round(
        df_battery["Total additional energy (kWh)"].sum() / 1000,
        2
    )
)

# 11. Display results
display(
    df_battery[
        [
            "Lijn",
            "Remise",
            "Model",
            "Aantal",
            "Totale km per bus",
            "Verbruik per bus (kWh)",
            "Capacity",
            "Battery equivalents",
            "Required battery charges",
            "Overnight charging sufficient",
            "Additional energy (kWh per bus)",
            "Additional charging sessions",
            "Buses requiring additional charging"
        ]
    ]
)


BATTERY CHARGING ANALYSIS
-------------------------
Total assigned buses: 293
Buses requiring additional charging: 54
Additional charging energy (MWh): 5.9


,Lijn,Remise,Model,Aantal,Totale km per bus,Verbruik per bus (kWh),Capacity,Battery equivalents,Required battery charges,Overnight charging sufficient,Additional energy (kWh per bus),Additional charging sessions,Buses requiring additional charging
0,2,Westraven,E-way 10.7,1,141.476378,130.158268,385.00,0.338073,1,True,0.000000,0,0
1,3,Westraven,GX437 ELEC,13,145.851126,303.856513,250.00,1.215426,2,False,53.856513,1,13
2,7,Westraven,GX437 ELEC,8,219.574083,457.446006,250.00,1.829784,2,False,207.446006,1,8
3,8,Westraven,GX437 ELEC,8,209.386730,436.222355,250.00,1.744889,2,False,186.222355,1,8
4,28,Westraven,Urbino 24 Electric,16,241.178219,653.036409,704.00,0.927609,1,True,0.000000,0,0
5,43,Westraven,Urbino 9 Electric,2,306.277713,245.022170,350.00,0.700063,1,True,0.000000,0,0
6,44,Nieuwegein,Urbino 9 Electric,2,354.087840,283.270272,350.00,0.809344,1,True,0.000000,0,0
7,57,Westraven,Urbino 9 Electric,2,212.021583,169.617266,350.00,0.484621,1,True,0.000000,0,0
8,81,Nieuwegein,Urbino 9 Electric,2,314.103068,251.282455,350.00,0.717950,1,True,0.000000,0,0
9,85,Nieuwegein,Urbino 15 Electric,13,232.225597,436.584123,470.00,0.928902,1,True,0.000000,0,0


In [ ]:

# STEP 7: Calculate overnight charging demand per depot

# 1. Start with the battery analysis
df_night_charging = df_battery.copy()  # start from the battery table

# 2. Calculate full overnight charging energy
# Assume every assigned bus receives one full battery charge

df_night_charging["Night energy per bus (kWh)"] = (  # one full battery per bus (upper estimate)
    df_night_charging["Capacity"]
)

df_night_charging["Total night energy (kWh)"] = (  # x number of buses
    df_night_charging["Aantal"]
    * df_night_charging["Night energy per bus (kWh)"]
)

# 3. Calculate overnight charging demand per depot
df_depot_charging = (  # add up per depot
    df_night_charging.groupby("Remise", as_index=False)
    .agg(
        Number_of_buses=("Aantal", "sum"),  # buses per depot
        Night_energy_kWh=("Total night energy (kWh)", "sum")  # night energy per depot
    )
)

# 4. Convert energy to MWh
df_depot_charging["Night_energy_MWh"] = (  # kWh -> MWh
    df_depot_charging["Night_energy_kWh"] / 1000
)

# 5. Calculate total charging demand
total_buses = int(df_depot_charging["Number_of_buses"].sum())  # all buses

total_energy_kWh = df_depot_charging["Night_energy_kWh"].sum()  # all night energy (kWh)

total_energy_MWh = total_energy_kWh / 1000  # -> MWh

# 6. Validate bus count
expected_buses = int(df_assignment_final["Aantal"].sum())  # buses in the final assignment

if total_buses != expected_buses:  # must be the same number
    raise ValueError(  # stop with an error message
        "Night charging bus count does not match assignments."
    )

# 7. Display summary
# Print the totals
print("OVERNIGHT CHARGING DEMAND")
print("-------------------------")  # print a check / summary line
print("Total buses:", total_buses)  # print a check / summary line
print("Total overnight energy (MWh):", round(total_energy_MWh, 2))  # print a check / summary line

# 8. Display demand per depot
display(
    df_depot_charging.round(2)
)

# 9. Display detailed demand per bus model and depot
df_night_charging_by_model = (  # the same, split by depot, maker, model and battery size
    df_night_charging.groupby(
        ["Remise", "Fabrikant", "Model", "Capacity"],
        as_index=False
    )
    .agg(
        Number_of_buses=("Aantal", "sum"),
        Night_energy_kWh=("Total night energy (kWh)", "sum")
    )
)

df_night_charging_by_model["Night_energy_MWh"] = (  # kWh -> MWh
    df_night_charging_by_model["Night_energy_kWh"] / 1000
)

display(  # show the table
    df_night_charging_by_model.sort_values(
        ["Remise", "Model"]
    ).round(2)
)


OVERNIGHT CHARGING DEMAND
-------------------------
Total buses: 293
Total overnight energy (MWh): 132.5


,Remise,Number_of_buses,Night_energy_kWh,Night_energy_MWh
0,Lage Weide,47,19860.86,19.86
1,Nieuwegein,56,26901.99,26.90
2,Westraven,190,85736.04,85.74


,Remise,Fabrikant,Model,Capacity,Number_of_buses,Night_energy_kWh,Night_energy_MWh
4,Lage Weide,Volvo,7900 EA,316.00,9,2844.00,2.84
3,Lage Weide,VDL,Citea LF-122 Electric,450.00,10,4500.00,4.50
0,Lage Weide,Iveco,E-way 10.7,385.00,2,770.00,0.77
5,Lage Weide,Yutong,U12 LE,465.99,14,6523.86,6.52
1,Lage Weide,Solaris,Urbino 18 Electric,443.00,11,4873.00,4.87
2,Lage Weide,Solaris,Urbino 9 Electric,350.00,1,350.00,0.35
10,Nieuwegein,Volvo,7900 EA,316.00,8,2528.00,2.53
9,Nieuwegein,VDL,Citea LF-122 Electric,450.00,2,900.00,0.90
11,Nieuwegein,Yutong,U12 LE,465.99,5,2329.95,2.33
12,Nieuwegein,Yutong,U18 LF,704.17,12,8450.04,8.45


In [ ]:

# STEP 8: Calculate overnight charging power demand per depot

# 1. Define charging windows in hours
charging_windows = [6, 8, 10]  # possible night charging windows (hours)

# 2. Start with the overnight energy demand per depot
df_power = df_depot_charging.copy()  # night energy per depot

# 3. Calculate average charging power for each charging window
for hours in charging_windows:  # for each window length

    # Average power in kW
    df_power[f"Power {hours}h (kW)"] = (  # average power = energy / hours
        df_power["Night_energy_kWh"] / hours
    )

    # Average power in MW
    df_power[f"Power {hours}h (MW)"] = (  # kW -> MW
        df_power[f"Power {hours}h (kW)"] / 1000
    )

# 4. Calculate the total average power across all depots
total_power = {  # total row: sums over all depots
    "Remise": "TOTAL",
    "Number_of_buses": df_power["Number_of_buses"].sum(),
    "Night_energy_kWh": df_power["Night_energy_kWh"].sum(),
    "Night_energy_MWh": df_power["Night_energy_MWh"].sum()
}

for hours in charging_windows:  # same power calculation for the total

    total_power[f"Power {hours}h (kW)"] = (  # energy / hours
        total_power["Night_energy_kWh"] / hours
    )

    total_power[f"Power {hours}h (MW)"] = (  # kW -> MW
        total_power[f"Power {hours}h (kW)"] / 1000
    )

# 5. Add total row
df_power = pd.concat(  # add the total row
    [df_power, pd.DataFrame([total_power])],
    ignore_index=True
)

# 6. Display results
print("OVERNIGHT CHARGING POWER DEMAND")
print("--------------------------------")  # print a check / summary line

display(df_power.round(2))  # show, rounded to 2 decimals


OVERNIGHT CHARGING POWER DEMAND
--------------------------------


,Remise,Number_of_buses,Night_energy_kWh,Night_energy_MWh,Power 6h (kW),Power 6h (MW),Power 8h (kW),Power 8h (MW),Power 10h (kW),Power 10h (MW)
0,Lage Weide,47,19860.86,19.86,3310.14,3.31,2482.61,2.48,1986.09,1.99
1,Nieuwegein,56,26901.99,26.90,4483.66,4.48,3362.75,3.36,2690.20,2.69
2,Westraven,190,85736.04,85.74,14289.34,14.29,10717.00,10.72,8573.60,8.57
3,TOTAL,293,132498.89,132.50,22083.15,22.08,16562.36,16.56,13249.89,13.25


In [ ]:
# Select buses requiring more than one battery charge

df_extra_charging = df_battery[  # rows where buses need more than one full charge per day
    df_battery["Required battery charges"] > 1
].copy()

df_extra_charging = df_extra_charging[  # keep the relevant columns
    [
        "Lijn",
        "Remise",
        "Fabrikant",
        "Model",
        "Aantal",
        "Totale km per bus",
        "Verbruik per bus (kWh)",
        "Capacity",
        "Required battery charges",
        "Additional charging sessions",
        "Additional energy (kWh per bus)",
        "Total additional energy (kWh)"
    ]
]

display(df_extra_charging)  # show them

,Lijn,Remise,Fabrikant,Model,Aantal,Totale km per bus,Verbruik per bus (kWh),Capacity,Required battery charges,Additional charging sessions,Additional energy (kWh per bus),Total additional energy (kWh)
1,3,Westraven,Heuliez,GX437 ELEC,13,145.851126,303.856513,250.0,2,1,53.856513,700.134668
2,7,Westraven,Heuliez,GX437 ELEC,8,219.574083,457.446006,250.0,2,1,207.446006,1659.568052
3,8,Westraven,Heuliez,GX437 ELEC,8,209.386730,436.222355,250.0,2,1,186.222355,1489.778839
12,1,Lage Weide,Solaris,Urbino 18 Electric,4,246.925313,546.939569,443.0,2,1,103.939569,415.758275
19,6,Lage Weide,Solaris,Urbino 18 Electric,2,351.228094,777.970228,443.0,2,1,334.970228,669.940457
65,65,Nieuwegein,Solaris,Urbino 18 Electric,4,205.618590,455.445178,443.0,2,1,12.445178,49.780712
66,66,Nieuwegein,Volvo,7900 EA,1,472.585126,472.585126,316.0,2,1,156.585126,156.585126
67,66,Nieuwegein,Solaris,Urbino 18 Electric,1,472.585126,1046.776054,443.0,3,2,603.776054,603.776054
71,74,Westraven,Solaris,Urbino 18 Electric,13,205.429961,455.027363,443.0,2,1,12.027363,156.355720


In [ ]:

# Model hourly charging demand for each depot

# 1. Define the overnight charging period
charging_start = 22  # charging starts at 22:00
charging_end = 6  # and ends at 06:00 the next morning

# 2. Calculate charging duration
charging_duration = (charging_end - charging_start) % 24  # hours in the window; % 24 handles midnight (6 - 22 = -16 -> 8)

if charging_duration == 0:  # start = end would mean no charging window
    raise ValueError("Charging duration must be between 1 and 23 hours.")  # stop with an error message

# 3. Create 24 hourly intervals for each depot
hours = pd.DataFrame({  # table with the 24 clock hours 0-23
    "hour_start": range(24)
})

df_hourly_charging = df_depot_charging.merge(  # every depot x every hour (3 x 24 rows)
    hours,
    how="cross"  # cross join: combine every row with every row
)

# 4. Identify the active charging hours
df_hourly_charging["Charging active"] = (  # True for the hours inside the window
    (df_hourly_charging["hour_start"] - charging_start) % 24  # hours since 22:00 (wraps around midnight)
    < charging_duration  # inside the window if less than the window length
)

# 5. Distribute overnight energy evenly across the charging period
df_hourly_charging["Charging demand (MW)"] = np.where(  # constant power during the window, 0 outside
    df_hourly_charging["Charging active"],  # if the hour is a charging hour:
    df_hourly_charging["Night_energy_MWh"] / charging_duration,  # power = night energy / number of hours
    0.0  # otherwise no charging
)

# 6. Calculate hourly charging energy
df_hourly_charging["Charging energy (MWh)"] = (  # energy in one hour = power x 1 h, so the same number
    df_hourly_charging["Charging demand (MW)"]
)

# 7. Validate total energy per depot
energy_check = (  # add up the hourly energy per depot
    df_hourly_charging.groupby("Remise", as_index=False)
    ["Charging energy (MWh)"].sum()
)

energy_check = energy_check.merge(  # next to the night energy it should equal
    df_depot_charging[["Remise", "Night_energy_MWh"]],
    on="Remise",
    validate="one_to_one"
)

if not np.allclose(  # the two must be equal (up to rounding)
    energy_check["Charging energy (MWh)"],
    energy_check["Night_energy_MWh"]
):
    raise ValueError("Charging energy does not match overnight demand.")  # stop with an error message

# 8. Create summary per depot
df_charging_summary = (  # one row per depot
    df_hourly_charging.groupby("Remise", as_index=False)
    .agg(
        Night_energy_MWh=("Charging energy (MWh)", "sum"),  # total night energy
        Charging_power_MW=("Charging demand (MW)", "max")  # charging power during the window
    )
)

# 9. Create hourly overview
df_charging_profile = df_hourly_charging.pivot(  # table: hours as rows, depots as columns
    index="hour_start",  # rows = hour of the day
    columns="Remise",  # columns = depots
    values="Charging demand (MW)"  # values = charging power
).reset_index()

# 10. Display results
print("OVERNIGHT CHARGING MODEL")
print("------------------------")  # print a check / summary line
print(f"Charging period: {charging_start:02d}:00 - {charging_end:02d}:00")  # e.g. 'Charging period: 22:00 - 06:00'
print("Charging duration:", charging_duration, "hours")  # number of charging hours

display(df_charging_summary.round(2))  # night energy and charging power per depot
display(df_charging_profile.round(2))  # charging power per hour and depot


OVERNIGHT CHARGING MODEL
------------------------
Charging period: 22:00 - 06:00
Charging duration: 8 hours


,Remise,Night_energy_MWh,Charging_power_MW
0,Lage Weide,19.86,2.48
1,Nieuwegein,26.90,3.36
2,Westraven,85.74,10.72


Remise,hour_start,Lage Weide,Nieuwegein,Westraven
0,0,2.48,3.36,10.72
1,1,2.48,3.36,10.72
2,2,2.48,3.36,10.72
3,3,2.48,3.36,10.72
4,4,2.48,3.36,10.72
5,5,2.48,3.36,10.72
6,6,0.00,0.00,0.00
7,7,0.00,0.00,0.00
8,8,0.00,0.00,0.00
9,9,0.00,0.00,0.00


In [ ]:

#Compare hourly charging demand with grid capacity

# 1. Load grid data
congestion_folder = Path("congestion data")  # folder with the datasets of the grid part

depot_grid = pd.read_csv(  # dataset 1: depots, supply areas and grid connections
    congestion_folder / "1_Depot_and_Grid_Supply_Area_Data.csv"
)

grid_scenarios = pd.read_csv(  # dataset 4: hourly area load and capacity per depot
    congestion_folder / "4_Bus_Charging_and_Grid_Comparison.csv"
)

# 2. Match depot names
depot_mapping = {  # depot names in this script -> names in the grid datasets
    "Westraven": "Utrecht - Westraven",
    "Nieuwegein": "Nieuwegein - Remiseweg",
    "Lage Weide": "Utrecht - Lage Weide"
}

charging = df_hourly_charging.copy()  # copy of the hourly charging profile

charging["bus_depot"] = charging["Remise"].map(  # add the grid-dataset name of each depot
    depot_mapping
)

if charging["bus_depot"].isna().any():  # a depot name that could not be translated
    raise ValueError("Unknown depot name in charging data.")  # stop with an error message

# 3. Prepare existing hourly grid load
# area_excl_depot_MW excludes the original modelled depot load

grid_hourly = grid_scenarios[  # keep date, hour, depot, area, area load without the depot and capacity
    [
        "date",
        "hour_start",
        "bus_depot",
        "supply_area_id",
        "area_excl_depot_MW",
        "capacity_MW"
    ]
].copy()

# Local date and clock hour as one label, e.g. "2025-11-27 17:00"
grid_hourly.insert(  # put it in front as the first column
    0,
    "datetime_local",
    grid_hourly["date"].astype(str)  # the date ...
    + " "
    + grid_hourly["hour_start"].astype(int).map("{:02d}:00".format)  # ... plus the clock hour
)

# Convert numeric values
for column in [  # the two numeric columns ...
    "area_excl_depot_MW",
    "capacity_MW"
]:
    grid_hourly[column] = pd.to_numeric(  # ... as numbers (text -> NaN)
        grid_hourly[column],
        errors="coerce"
    )

# 4. Add grid connection capacity
connections = depot_grid[  # grid connection of each depot (one row per depot)
    [
        "bus_depot",
        "supply_area_id",
        "connection_MW"
    ]
].drop_duplicates("bus_depot").rename(  # one row per depot
    columns={"supply_area_id": "depot_supply_area_id"}  # other name, so it does not clash with the hourly table
).copy()

connections["connection_MW"] = pd.to_numeric(  # connection as a number
    connections["connection_MW"],
    errors="coerce"
)

grid_hourly = grid_hourly.merge(  # add the connection to every hour
    connections,
    on="bus_depot",
    how="left",
    validate="many_to_one"  # many hours, one connection per depot (checked by pandas)
)

# Verify supply area identifiers
mismatch = grid_hourly[  # rows where the two datasets name a different supply area
    grid_hourly["supply_area_id"]  # supply area in the hourly data (dataset 4)
    != grid_hourly["depot_supply_area_id"]  # supply area of the depot (dataset 1)
]

if not mismatch.empty:  # stop if they do not agree
    raise ValueError(  # stop with an error message
        "Grid supply area identifiers do not match."
    )

# 5. Add our calculated charging demand
df_grid_hourly_comparison = grid_hourly.merge(  # add the charging power of the same depot and clock hour to every hour of the year
    charging[
        [
            "bus_depot",
            "hour_start",
            "Charging demand (MW)"
        ]
    ],
    on=["bus_depot", "hour_start"],  # match on depot and hour of the day
    how="inner",  # keep only depots that are in both tables
    validate="many_to_one"  # 365 days per depot and hour, one charging value
)

# 6. Check missing depot coverage
missing_depots = (  # depots with charging demand but without grid data
    set(charging["bus_depot"])
    - set(df_grid_hourly_comparison["bus_depot"])
)

if missing_depots:  # warn if there are any
    print("WARNING: No grid data available for:")  # print a check / summary line
    print(sorted(missing_depots))  # print a check / summary line

# Check missing values
required_columns = [  # columns that must be filled in
    "area_excl_depot_MW",
    "capacity_MW",
    "connection_MW",
    "Charging demand (MW)"
]

if df_grid_hourly_comparison[required_columns].isna().any().any():  # stop if any of them is empty
    raise ValueError(  # stop with an error message
        "Missing values in grid or charging data."
    )

# 7. Calculate the new total grid load
df_grid_hourly_comparison["Total grid load (MW)"] = (  # area load without the depot + our charging demand
    df_grid_hourly_comparison["area_excl_depot_MW"]
    + df_grid_hourly_comparison["Charging demand (MW)"]
)

# 8. Calculate remaining supply area capacity
df_grid_hourly_comparison["Grid headroom (MW)"] = (  # capacity - total load (negative = over capacity)
    df_grid_hourly_comparison["capacity_MW"]
    - df_grid_hourly_comparison["Total grid load (MW)"]
)

# 9. Calculate remaining depot connection capacity
df_grid_hourly_comparison["Connection headroom (MW)"] = (  # connection - charging demand (negative = connection too small)
    df_grid_hourly_comparison["connection_MW"]
    - df_grid_hourly_comparison["Charging demand (MW)"]
)

# 10. Identify capacity exceedances
df_grid_hourly_comparison["Grid exceeded"] = (  # True if the area is over its capacity
    df_grid_hourly_comparison["Grid headroom (MW)"] < 0
)

df_grid_hourly_comparison["Connection exceeded"] = (  # True if the depot connection is overloaded
    df_grid_hourly_comparison["Connection headroom (MW)"] < 0
)

# 11. Summarise results per depot
df_grid_summary = (  # one row per depot
    df_grid_hourly_comparison.groupby(
        "bus_depot",
        as_index=False
    )
    .agg(
        Hours_analysed=("datetime_local", "count"),  # number of hours (8,760 = one year)
        Grid_exceeded_hours=("Grid exceeded", "sum"),  # hours over the area capacity (True counts as 1)
        Connection_exceeded_hours=("Connection exceeded", "sum"),  # hours over the connection
        Minimum_grid_headroom_MW=(  # smallest remaining area capacity
            "Grid headroom (MW)", "min"
        ),
        Minimum_connection_headroom_MW=(  # smallest remaining connection capacity
            "Connection headroom (MW)", "min"
        ),
        Maximum_charging_demand_MW=(  # highest charging power
            "Charging demand (MW)", "max"
        ),
        Connection_capacity_MW=("connection_MW", "first")  # size of the connection
    )
)

# 12. Add percentage of hours exceeding capacity
df_grid_summary["Grid_exceeded_pct"] = (  # share of all hours over the area capacity
    df_grid_summary["Grid_exceeded_hours"]
    / df_grid_summary["Hours_analysed"] * 100
)

df_grid_summary["Connection_exceeded_pct"] = (  # share of all hours over the connection
    df_grid_summary["Connection_exceeded_hours"]
    / df_grid_summary["Hours_analysed"] * 100
)

# 13. Display results
print("HOURLY GRID CAPACITY COMPARISON")
print("--------------------------------")  # print a check / summary line
print(  # print a check / summary line
    f"Charging period: "
    f"{charging_start:02d}:00 - {charging_end:02d}:00"
)

print("\nSummary per depot:")  # print a check / summary line
display(df_grid_summary.round(2))  # summary per depot

print("\nHourly comparison:")  # print a check / summary line
display(df_grid_hourly_comparison.head(24).round(2))  # the first 24 hours as an example

print("\nDepots without grid data:")  # print a check / summary line
print(sorted(missing_depots))  # print a check / summary line


In [ ]:

# STEP 12: Summarise charging feasibility per depot

import pandas as pd  # already imported above
import numpy as np  # already imported above
from pathlib import Path  # already imported above

# 1. Load grid capacity and congestion data
congestion_folder = Path("congestion data")  # same folder as in the previous cell

grid_capacity_data = pd.read_csv(  # dataset 2: capacity, peak and queue per supply area (long table)
    congestion_folder / "2_Grid_Capacity_and_Congestion_Data.csv"
)

grid_capacity_data["value_used"] = pd.to_numeric(  # values as numbers (text such as a status -> NaN)
    grid_capacity_data["value_used"],
    errors="coerce"
)

# 2. Convert grid metrics into columns
capacity_summary = grid_capacity_data.pivot_table(  # one row per depot, one column per metric
    index="bus_depot",  # rows = depots
    columns="metric",  # columns = the metric names
    values="value_used",  # cells = the value used
    aggfunc="first"  # one value per depot and metric
).reset_index()

capacity_summary.columns.name = None  # remove the leftover 'metric' label above the columns

metric_names = {  # metric names in dataset 2 -> column names used below
    "present_transport_capacity_offtake": "present_transport_capacity_MW",  # capacity of the supply area
    "required_transport_capacity_offtake_peak": "required_peak_transport_capacity_MW",  # load at the peak moment
    "headroom_at_peak": "simple_peak_difference_MW",  # capacity - peak
    "queue_offtake_MW": "offtake_queue_MW"  # capacity that others are already waiting for
}

capacity_summary = capacity_summary[  # keep only those metrics ...
    ["bus_depot"] + list(metric_names)
].rename(columns=metric_names).copy()  # ... under the names used below

# 3. Calculate remaining capacity after the queue
capacity_summary["Headroom after queue (MW)"] = (  # (capacity - peak) - queue: room left once the waiting requests are connected
    capacity_summary["simple_peak_difference_MW"]
    - capacity_summary["offtake_queue_MW"]
)

# 4. Calculate hourly exceedances during charging
hourly = df_grid_hourly_comparison.copy()  # copy of the hourly comparison

hourly["Charging active"] = (  # True in the hours with charging
    hourly["Charging demand (MW)"] > 0
)

hourly["Grid exceeded during charging"] = (  # area over capacity AND charging at that moment
    hourly["Grid exceeded"]
    & hourly["Charging active"]
)

hourly["Connection exceeded during charging"] = (  # connection overloaded AND charging at that moment
    hourly["Connection exceeded"]
    & hourly["Charging active"]
)

hourly_summary = (  # one row per depot
    hourly.groupby("bus_depot", as_index=False)
    .agg(
        Hours_analysed=("datetime_local", "count"),  # all hours
        Charging_hours=("Charging active", "sum"),  # hours with charging
        Grid_exceeded_hours=(  # charging hours over the area capacity
            "Grid exceeded during charging", "sum"
        ),
        Connection_exceeded_hours=(  # charging hours over the connection
            "Connection exceeded during charging", "sum"
        ),
        Minimum_grid_headroom_MW=(  # smallest remaining area capacity
            "Grid headroom (MW)", "min"
        ),
        Minimum_connection_headroom_MW=(  # smallest remaining connection capacity
            "Connection headroom (MW)", "min"
        ),
        Connection_capacity_MW=(  # size of the connection
            "connection_MW", "first"
        )
    )
)

# 5. Connect results to charging demand per depot
depot_mapping = {  # same name translation as in the previous cell
    "Westraven": "Utrecht - Westraven",
    "Nieuwegein": "Nieuwegein - Remiseweg",
    "Lage Weide": "Utrecht - Lage Weide"
}

df_grid_final = df_charging_summary.copy()  # start from energy and power per depot

df_grid_final["bus_depot"] = (  # add the grid-dataset name
    df_grid_final["Remise"].map(depot_mapping)
)

df_grid_final = df_grid_final.merge(  # + number of buses
    df_depot_charging[
        ["Remise", "Number_of_buses"]
    ],
    on="Remise",
    how="left",
    validate="one_to_one"
)

df_grid_final = df_grid_final.merge(  # + hourly results
    hourly_summary,
    on="bus_depot",
    how="left",
    validate="one_to_one"
)

df_grid_final = df_grid_final.merge(  # + capacity, peak and queue
    capacity_summary,
    on="bus_depot",
    how="left",
    validate="one_to_one"
)

# 6. Check complete coverage
if df_grid_final[  # stop if a depot misses grid data
    [
        "Hours_analysed",
        "Connection_capacity_MW",
        "present_transport_capacity_MW"
    ]
].isna().any().any():
    raise ValueError(  # stop with an error message
        "Missing grid capacity data for one or more depots."
    )

# 7. Calculate connection margin
df_grid_final["Connection margin (MW)"] = (  # connection - charging power (negative = connection too small)
    df_grid_final["Connection_capacity_MW"]
    - df_grid_final["Charging_power_MW"]
)

# 8. Calculate percentage of charging hours with exceedances
df_grid_final["Grid exceeded (%)"] = (  # share of the charging hours over the area capacity
    df_grid_final["Grid_exceeded_hours"]
    / df_grid_final["Charging_hours"] * 100
)

df_grid_final["Connection exceeded (%)"] = (  # share of the charging hours over the connection
    df_grid_final["Connection_exceeded_hours"]
    / df_grid_final["Charging_hours"] * 100
)

# 9. Add data source identification
capacity_basis = grid_capacity_data[  # is the capacity of the supply area published or estimated?
    grid_capacity_data["metric"] == "present_transport_capacity_offtake"
].set_index("bus_depot")["basis"]  # OBSERVED = published, ASSUMED = estimated

df_grid_final["Data basis"] = df_grid_final["bus_depot"].map(capacity_basis).map({  # readable label per depot
    "OBSERVED": "Published capacity (Netbeheer open data)",
    "ASSUMED": "Estimated capacity (not published, see dataset 2)"
})

# 10. Determine modelled charging feasibility
df_grid_final["Charging feasibility"] = np.select(  # the first condition that is true decides the label
    [
        df_grid_final["Connection_exceeded_hours"] > 0,  # 1. connection overloaded in any charging hour
        df_grid_final["Grid_exceeded_hours"] > 0  # 2. area over capacity in any charging hour
    ],
    [
        "Connection capacity exceeded",  # label for condition 1
        "Grid capacity exceeded"  # label for condition 2
    ],
    default="Within modelled capacity"  # neither: charging fits
)

# 11. Select columns for final report
df_grid_final = df_grid_final[  # the columns for the report, in this order
    [
        "Remise",
        "Number_of_buses",
        "Night_energy_MWh",
        "Charging_power_MW",
        "Connection_capacity_MW",
        "Connection margin (MW)",
        "present_transport_capacity_MW",
        "required_peak_transport_capacity_MW",
        "simple_peak_difference_MW",
        "offtake_queue_MW",
        "Headroom after queue (MW)",
        "Charging_hours",
        "Grid_exceeded_hours",
        "Connection_exceeded_hours",
        "Grid exceeded (%)",
        "Connection exceeded (%)",
        "Minimum_grid_headroom_MW",
        "Charging feasibility",
        "Data basis"
    ]
]

# 12. Display final results
print("FINAL GRID CAPACITY SUMMARY")
print("---------------------------")  # print a check / summary line
print(  # print a check / summary line
    f"Charging period: "
    f"{charging_start:02d}:00 - {charging_end:02d}:00"
)

display(df_grid_final.round(2))  # final table per depot


In [ ]:

# STEP 13: Export all bus, charging and grid analysis results


output_file = Path("Utrecht_Bus_Energy_Analysis.xlsx")  # name of the Excel output file

# 1. Define all output sheets
sheets = {  # sheet name -> table to write
    # Bus line and depot analysis
    "Bus lines": bus_lines_final,  # line data with depot and empty km
    "Depot distances": depot_distances,  # empty km per line and depot
    "Depot locations": depot_locations,  # coordinates of the depots

    # Bus model assignments
    "Bus assignments": df_assignment_final,  # buses per line and model
    "Fleet overview": df_fleet_final,  # fleet used and left over per model
    "Bus shortages": df_shortages,  # lines without enough buses
    "Excluded lines": excluded_lines,  # lines without a known model

    # Daily energy and battery analysis
    "Daily energy": df_verbruik,  # km and kWh per bus per day
    "Battery analysis": df_battery,  # is one battery enough per day?
    "Extra charging": df_extra_charging,  # buses that need an extra charge

    # Overnight charging analysis
    "Night charging details": df_night_charging_by_model,  # night energy per depot and model
    "Night charging depots": df_depot_charging,  # night energy per depot
    "Charging power": df_power,  # average power for 6, 8 and 10 hour windows

    # NEW: Charging period and grid congestion analysis
    "Hourly charging profile": df_hourly_charging,  # charging power per depot and hour
    "Charging summary": df_charging_summary,  # energy and power per depot in the window
    "Grid capacity summary": df_grid_final,  # feasibility per depot
    "Hourly grid comparison": df_grid_hourly_comparison  # every hour of the year: load, charging and headroom
}

# 2. Check that every result is available
for sheet_name, dataframe in sheets.items():  # check every table ...

    if not isinstance(dataframe, pd.DataFrame):  # ... is really a DataFrame
        raise ValueError(  # stop with an error message
            f"{sheet_name} is not a valid DataFrame."
        )

# 3. Validate final bus counts
assigned_buses = int(  # buses in the assignment
    df_assignment_final["Aantal"].sum()
)

night_buses = int(  # buses in the night charging table
    df_depot_charging["Number_of_buses"].sum()
)

if assigned_buses != night_buses:  # must be the same
    raise ValueError(  # stop with an error message
        "Assigned bus count does not match night charging."
    )

# 4. Validate charging energy
total_night_energy = (  # night energy of all depots
    df_depot_charging["Night_energy_MWh"].sum()
)

total_hourly_energy = (  # sum of the hourly charging energy
    df_hourly_charging["Charging energy (MWh)"].sum()
)

if not pd.isna(total_hourly_energy):  # only if the hourly table has values
    if abs(total_night_energy - total_hourly_energy) > 0.001:  # must be equal (tolerance 0.001 MWh)
        raise ValueError(  # stop with an error message
            "Hourly charging energy does not match overnight energy."
        )

# 5. Validate grid data coverage
expected_depots = set(  # depots in the final summary
    df_grid_final["Remise"]
)

actual_depots = set(  # depots in the hourly grid comparison
    df_grid_hourly_comparison["bus_depot"]
)

depot_mapping = {  # same name translation as above
    "Westraven": "Utrecht - Westraven",
    "Nieuwegein": "Nieuwegein - Remiseweg",
    "Lage Weide": "Utrecht - Lage Weide"
}

expected_grid_depots = {  # summary depots translated to the grid-dataset names
    depot_mapping.get(depot, depot)
    for depot in expected_depots
}

if expected_grid_depots != actual_depots:  # both must contain the same depots
    raise ValueError(  # stop with an error message
        "Grid comparison does not contain all expected depots."
    )

# 6. Export all sheets
with pd.ExcelWriter(  # open the Excel file (closed automatically at the end)
    output_file,
    engine="openpyxl"
) as writer:

    for sheet_name, dataframe in sheets.items():  # every table

        export_data = dataframe.copy()  # copy, so the original table is not changed

        # Convert list columns to Excel-compatible text
        for column in export_data.columns:  # every column

            export_data[column] = export_data[column].apply(  # lists become text 'A; B'; other values stay
                lambda value: "; ".join(map(str, value))
                if isinstance(value, (list, tuple, set))
                else value
            )

        export_data.to_excel(  # write the table to its own sheet
            writer,
            sheet_name=sheet_name,
            index=False
        )

        # Improve worksheet readability
        worksheet = writer.sheets[sheet_name]  # the sheet just written

        worksheet.freeze_panes = "A2"  # keep the header row visible when scrolling
        worksheet.auto_filter.ref = worksheet.dimensions  # filter buttons on the header row

        for column_cells in worksheet.columns:  # every column of the sheet

            column_letter = (  # its letter (A, B, ...)
                column_cells[0].column_letter
            )

            max_length = max(  # longest text in the column
                len(str(cell.value))
                if cell.value is not None
                else 0
                for cell in column_cells
            )

            worksheet.column_dimensions[column_letter].width = (  # width between 12 and 45 characters
                min(
                    max(max_length + 2, 12),
                    45
                )
            )

# 7. Confirm export
print("EXCEL EXPORT COMPLETED")
print("----------------------")  # print a check / summary line
print("File:", output_file.resolve())  # print a check / summary line
print("Number of sheets:", len(sheets))  # print a check / summary line
print("Assigned buses:", assigned_buses)  # print a check / summary line

print(  # print a check / summary line
    "Shortage:",
    int(df_shortages["Niet toegewezen"].sum())
)

print(  # print a check / summary line
    "Overnight charging energy (MWh):",
    round(total_night_energy, 2)
)

print(  # print a check / summary line
    f"Charging period: "
    f"{charging_start:02d}:00 - {charging_end:02d}:00"
)

print("\nGrid comparison depots:")  # print a check / summary line
print(sorted(actual_depots))  # print a check / summary line


EXCEL EXPORT COMPLETED
----------------------
File: C:\Users\danie\Documents\MSC Transport Infrastructure & Logistics\Jaar 1\Q1\TIL programming\Assignment\Utrecht_Bus_Energy_Analysis.xlsx
Number of sheets: 17
Assigned buses: 293
Shortage: 12
Overnight charging energy (MWh): 132.5
Charging period: 22:00 - 06:00

Grid comparison depots:
['Nieuwegein - Remiseweg', 'Utrecht - Lage Weide', 'Utrecht - Westraven']
